# E-Commerce Sales Performance and Customer Behavior Analysis
## Using Data Analytics and AI

---

**Dataset:** `ecommerce_customer_behavior_dataset_v2.csv`  
**Records:** 17,049 rows | **Columns:** 18 | **Unique Customers:** 5,000  
**Date Range:** January 1, 2023 – March 25, 2024  

**Notebook Sections:**
1. Data Loading, Cleaning & Feature Engineering ← *you are here*
2. KPI Computation
3. Exploratory Data Analysis — Univariate
4. Exploratory Data Analysis — Bivariate & Multivariate
5. Time Series Analysis
6. Customer Behaviour Analysis
7. Sales & Product Analysis
8. Delivery & Satisfaction Analysis
9. AI-Assisted Insight Framework
10. Streamlit Dashboard Code

> **Note:** All findings and outputs in this notebook are derived solely from
> calculations performed on the actual dataset. No results are assumed or invented
> prior to analysis. Run cells top-to-bottom in order.

---
## Section 1 — Data Loading, Cleaning & Feature Engineering

### 1.1 — Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings

# ── Core data processing ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Statistical analysis ──────────────────────────────────────────────────────
from scipy import stats

# ── Display settings ──────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 60)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})
sns.set_palette('tab10')

# ── Output directories ────────────────────────────────────────────────────────
os.makedirs('assets', exist_ok=True)
os.makedirs('report', exist_ok=True)

print('All imports successful.')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

### 1.2 — Load Dataset

In [ ]:
# ── Load from project root (relative path for portability) ───────────────────
DATA_FILE = 'ecommerce_customer_behavior_dataset_v2.csv'

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Dataset not found: '{DATA_FILE}'.\n"
        "Place the CSV in the same directory as this notebook."
    )

df = pd.read_csv(DATA_FILE)

print(f'Loaded  : {DATA_FILE}')
print(f'Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')

### 1.3 — Basic Dataset Inspection

In [ ]:
# ── Column names and data types ───────────────────────────────────────────────
print('── Column names and dtypes ──────────────────────────────────')
print(df.dtypes.to_string())

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print('── Missing values per column ───────────────────────────────')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')

In [ ]:
# ── Unique value counts per column ───────────────────────────────────────────
print('── Unique value counts ─────────────────────────────────────')
for col in df.columns:
    print(f'  {col:<32} {df[col].nunique():>6} unique values')

In [ ]:
# ── First 3 rows ─────────────────────────────────────────────────────────────
print('── First 3 rows ────────────────────────────────────────────')
df.head(3)

In [ ]:
# ── Numerical column statistics ───────────────────────────────────────────────
print('── Numerical column statistics ─────────────────────────────')
df.describe()

### 1.4 — Date Column Conversion

In [ ]:
# ── Convert Date string → datetime64 ─────────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'])

print(f'Date dtype after conversion : {df["Date"].dtype}')
print(f'Earliest date               : {df["Date"].min().date()}')
print(f'Latest date                 : {df["Date"].max().date()}')
print(f'Date range (days)           : {(df["Date"].max() - df["Date"].min()).days}')

### 1.5 — Derived Feature Engineering

In [ ]:
# ── Time-based features ───────────────────────────────────────────────────────
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter']    = df['Date'].dt.quarter
df['Year_Month'] = df['Date'].dt.to_period('M')

# ── Discount features ─────────────────────────────────────────────────────────
# Has_Discount: boolean flag — True when a discount was applied
df['Has_Discount'] = df['Discount_Amount'] > 0

# Discount_Rate_Pct: discount as a percentage of gross order value
# Guard against division by zero (Unit_Price * Quantity should never be 0)
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(
    gross > 0,
    (df['Discount_Amount'] / gross) * 100,
    0.0
)

# ── Age group ─────────────────────────────────────────────────────────────────
age_bins   = [17, 25, 35, 45, 55, 75]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-75']
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=age_bins,
    labels=age_labels,
    right=True
)

# ── Customer-level aggregates (merged back to order level) ────────────────────
cust_orders = (
    df.groupby('Customer_ID')['Order_ID']
    .count()
    .rename('Orders_Per_Customer')
)
cust_spend = (
    df.groupby('Customer_ID')['Total_Amount']
    .sum()
    .rename('Customer_Total_Spend')
)
df = df.merge(cust_orders, on='Customer_ID', how='left')
df = df.merge(cust_spend,  on='Customer_ID', how='left')

# ── High-value order flag (above 75th percentile of Total_Amount) ─────────────
p75 = df['Total_Amount'].quantile(0.75)
df['High_Value_Order'] = df['Total_Amount'] > p75

print('Derived features created:')
new_cols = [
    'Year', 'Month', 'Month_Name', 'Quarter', 'Year_Month',
    'Has_Discount', 'Discount_Rate_Pct', 'Age_Group',
    'Orders_Per_Customer', 'Customer_Total_Spend', 'High_Value_Order'
]
for c in new_cols:
    print(f'  + {c:<28} dtype: {df[c].dtype}')

print(f'\nDataFrame shape after engineering: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'75th percentile Total_Amount (High_Value_Order threshold): {p75:,.2f}')

### 1.6 — Data Validation & Assertions

In [ ]:
# ── Validation checks — all assertions must pass before proceeding ────────────
errors = []

# 1. Shape
assert df.shape[0] == 17049, 'Row count mismatch.'

# 2. No missing values in original 18 columns
original_cols = [
    'Order_ID', 'Customer_ID', 'Date', 'Age', 'Gender', 'City',
    'Product_Category', 'Unit_Price', 'Quantity', 'Discount_Amount',
    'Total_Amount', 'Payment_Method', 'Device_Type',
    'Session_Duration_Minutes', 'Pages_Viewed', 'Is_Returning_Customer',
    'Delivery_Time_Days', 'Customer_Rating'
]
for col in original_cols:
    assert df[col].isnull().sum() == 0, f'Unexpected nulls in column: {col}'

# 3. No duplicate Order_IDs
assert df['Order_ID'].duplicated().sum() == 0, 'Duplicate Order_IDs found.'

# 4. Date dtype is datetime
assert pd.api.types.is_datetime64_any_dtype(df['Date']), 'Date column is not datetime.'

# 5. Total_Amount == (Unit_Price * Quantity) - Discount_Amount (rounded to 2dp)
calculated = (df['Unit_Price'] * df['Quantity'] - df['Discount_Amount']).round(2)
mismatch = (calculated != df['Total_Amount'].round(2)).sum()
assert mismatch == 0, f'Total_Amount derivation mismatch: {mismatch} rows.'

# 6. No negative or zero prices
assert (df['Unit_Price'] > 0).all(), 'Unit_Price contains zero or negative values.'

# 7. No negative Total_Amount
assert (df['Total_Amount'] > 0).all(), 'Total_Amount contains zero or negative values.'

# 8. Quantity within expected range 1–5
assert df['Quantity'].between(1, 5).all(), 'Quantity out of expected range 1–5.'

# 9. Customer_Rating within 1–5
assert df['Customer_Rating'].between(1, 5).all(), 'Customer_Rating out of range 1–5.'

# 10. Delivery_Time_Days >= 1
assert (df['Delivery_Time_Days'] >= 1).all(), 'Delivery_Time_Days contains values < 1.'

# 11. Age within 18–75
assert df['Age'].between(18, 75).all(), 'Age values outside expected range 18–75.'

# 12. Discount_Amount >= 0
assert (df['Discount_Amount'] >= 0).all(), 'Discount_Amount contains negative values.'

# 13. Gender values
valid_genders = {'Female', 'Male', 'Other'}
assert set(df['Gender'].unique()).issubset(valid_genders), \
    f'Unexpected Gender values: {set(df["Gender"].unique()) - valid_genders}'

# 14. Is_Returning_Customer is boolean
assert df['Is_Returning_Customer'].dtype == bool, \
    'Is_Returning_Customer is not boolean dtype.'

# 15. Unique customer count
assert df['Customer_ID'].nunique() == 5000, 'Unique customer count is not 5,000.'

print('All 15 validation checks passed. ✓')
print('Dataset is clean and ready for analysis.')

In [ ]:
# ── Summary of final working DataFrame ───────────────────────────────────────
print('── Final DataFrame overview ────────────────────────────────')
print(f'Shape            : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range       : {df["Date"].min().date()}  →  {df["Date"].max().date()}')
print(f'Unique customers : {df["Customer_ID"].nunique():,}')
print(f'Unique orders    : {df["Order_ID"].nunique():,}')
print(f'Cities           : {sorted(df["City"].unique())}')
print(f'Categories       : {sorted(df["Product_Category"].unique())}')
print(f'Payment methods  : {sorted(df["Payment_Method"].unique())}')
print(f'Device types     : {sorted(df["Device_Type"].unique())}')
print(f'Gender values    : {sorted(df["Gender"].unique())}')
print(f'Age range        : {df["Age"].min()}  –  {df["Age"].max()}')
print()
print('Section 1 complete. Proceed to Section 2 — KPI Computation.')

---
## Section 2 — KPI Computation

All KPIs are computed directly from the dataset. No values are assumed or invented.
Formulas are stated for every KPI. Results reflect the full dataset unless otherwise noted.

### 2.1 — Overall KPIs

In [ ]:
# ── Rebuild working DataFrame (re-run Section 1 pipeline inline) ────────────
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Derived features required for KPIs
df['Year']          = df['Date'].dt.year
df['Month']         = df['Date'].dt.month
df['Month_Name']    = df['Date'].dt.strftime('%b')
df['Quarter']       = df['Date'].dt.quarter
df['Year_Month']    = df['Date'].dt.to_period('M')
df['Has_Discount']  = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross) * 100, 0.0)
age_bins   = [17, 25, 35, 45, 55, 75]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-75']
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=True)
cust_orders = df.groupby('Customer_ID')['Order_ID'].count().rename('Orders_Per_Customer')
cust_spend  = df.groupby('Customer_ID')['Total_Amount'].sum().rename('Customer_Total_Spend')
df = df.merge(cust_orders, on='Customer_ID', how='left')
df = df.merge(cust_spend,  on='Customer_ID', how='left')
p75 = df['Total_Amount'].quantile(0.75)
df['High_Value_Order'] = df['Total_Amount'] > p75

print('Working DataFrame ready: {:,} rows x {} columns'.format(df.shape[0], df.shape[1]))

In [ ]:
# ── KPI 1: Total Revenue ─────────────────────────────────────────────────────
# Formula: SUM(Total_Amount)
total_revenue = df['Total_Amount'].sum()

# ── KPI 2: Total Orders ───────────────────────────────────────────────────────
# Formula: COUNT(Order_ID)  — all rows are unique orders
total_orders = len(df)

# ── KPI 3: Unique Customers ───────────────────────────────────────────────────
# Formula: COUNT(DISTINCT Customer_ID)
unique_customers = df['Customer_ID'].nunique()

# ── KPI 4: Average Order Value (AOV) ─────────────────────────────────────────
# Formula: SUM(Total_Amount) / COUNT(Order_ID)
aov = total_revenue / total_orders

# ── KPI 5: Total Quantity Sold ────────────────────────────────────────────────
# Formula: SUM(Quantity)
total_qty = df['Quantity'].sum()

# ── KPI 6: Revenue per Customer ───────────────────────────────────────────────
# Formula: Total Revenue / Unique Customers
revenue_per_customer = total_revenue / unique_customers

# ── KPI 7: Returning Customer Rate ────────────────────────────────────────────
# Formula: COUNT(Is_Returning_Customer == True) / COUNT(Order_ID) * 100
returning_orders = df['Is_Returning_Customer'].sum()
returning_rate   = (returning_orders / total_orders) * 100

# ── KPI 8: Average Delivery Time ──────────────────────────────────────────────
# Formula: MEAN(Delivery_Time_Days)
avg_delivery = df['Delivery_Time_Days'].mean()

# ── KPI 9: Average Customer Rating ───────────────────────────────────────────
# Formula: MEAN(Customer_Rating)
avg_rating = df['Customer_Rating'].mean()

# ── KPI 10: Discount Penetration Rate ────────────────────────────────────────
# Formula: COUNT(Has_Discount == True) / COUNT(Order_ID) * 100
discounted_orders   = df['Has_Discount'].sum()
discount_penetration = (discounted_orders / total_orders) * 100

# ── KPI 11: Average Discount Amount (on discounted orders only) ───────────────
# Formula: MEAN(Discount_Amount) WHERE Has_Discount == True
avg_discount_when_applied = df.loc[df['Has_Discount'], 'Discount_Amount'].mean()

# ── KPI 12: Low-Rating Rate (ratings 1 or 2) ─────────────────────────────────
# Formula: COUNT(Customer_Rating <= 2) / COUNT(Order_ID) * 100
low_rating_rate = (df['Customer_Rating'].le(2).sum() / total_orders) * 100

# ── KPI 13: Avg Orders per Customer ──────────────────────────────────────────
# Formula: Total Orders / Unique Customers
avg_orders_per_customer = total_orders / unique_customers

# ── KPI 14: High-Value Order Rate ────────────────────────────────────────────
# Formula: COUNT(High_Value_Order == True) / COUNT(Order_ID) * 100
hvo_rate = (df['High_Value_Order'].sum() / total_orders) * 100

# ── Print KPI Summary Table ───────────────────────────────────────────────────
kpi_rows = [
    ('Total Revenue',                    'SUM(Total_Amount)',                                               '{:,.2f}'.format(total_revenue)),
    ('Total Orders',                     'COUNT(Order_ID)',                                                 '{:,}'.format(total_orders)),
    ('Unique Customers',                 'COUNT(DISTINCT Customer_ID)',                                     '{:,}'.format(unique_customers)),
    ('Average Order Value (AOV)',        'SUM(Total_Amount) / COUNT(Order_ID)',                             '{:,.2f}'.format(aov)),
    ('Total Quantity Sold',              'SUM(Quantity)',                                                   '{:,}'.format(total_qty)),
    ('Revenue per Customer',             'Total Revenue / Unique Customers',                                '{:,.2f}'.format(revenue_per_customer)),
    ('Returning Customer Rate',          'COUNT(Is_Returning_Customer=True) / Total Orders * 100',          '{:.2f}%'.format(returning_rate)),
    ('Returning Customer Orders',        'COUNT(Is_Returning_Customer=True)',                               '{:,}'.format(int(returning_orders))),
    ('Average Delivery Time (days)',     'MEAN(Delivery_Time_Days)',                                        '{:.2f}'.format(avg_delivery)),
    ('Average Customer Rating',          'MEAN(Customer_Rating)',                                           '{:.2f} / 5'.format(avg_rating)),
    ('Discount Penetration Rate',        'COUNT(Has_Discount=True) / Total Orders * 100',                   '{:.2f}%'.format(discount_penetration)),
    ('Avg Discount (when applied)',      'MEAN(Discount_Amount) WHERE Has_Discount=True',                   '{:,.2f}'.format(avg_discount_when_applied)),
    ('Low-Rating Rate (rating <= 2)',    'COUNT(Customer_Rating<=2) / Total Orders * 100',                  '{:.2f}%'.format(low_rating_rate)),
    ('Avg Orders per Customer',          'Total Orders / Unique Customers',                                 '{:.2f}'.format(avg_orders_per_customer)),
    ('High-Value Order Rate (>p75)',     'COUNT(Total_Amount > 75th pct) / Total Orders * 100',             '{:.2f}%'.format(hvo_rate)),
]

print('{:<35} {:<48} {}'.format('KPI', 'Formula', 'Value'))
print('-' * 100)
for name, formula, value in kpi_rows:
    print('{:<35} {:<48} {}'.format(name, formula, value))

### 2.2 — KPIs by Product Category

In [ ]:
# ── Revenue, Orders, AOV, Avg Rating, Avg Delivery by Category ───────────────
cat_kpi = (
    df.groupby('Product_Category', observed=True)
    .agg(
        Total_Revenue    =('Total_Amount',        'sum'),
        Total_Orders     =('Order_ID',            'count'),
        Total_Qty        =('Quantity',            'sum'),
        Avg_Order_Value  =('Total_Amount',        'mean'),
        Avg_Unit_Price   =('Unit_Price',          'mean'),
        Avg_Rating       =('Customer_Rating',     'mean'),
        Avg_Delivery_Days=('Delivery_Time_Days',  'mean'),
        Discount_Pct_Orders=('Has_Discount',      'mean'),
    )
    .sort_values('Total_Revenue', ascending=False)
    .reset_index()
)
cat_kpi['Discount_Pct_Orders'] = cat_kpi['Discount_Pct_Orders'] * 100
cat_kpi['Revenue_Share_Pct']   = cat_kpi['Total_Revenue'] / cat_kpi['Total_Revenue'].sum() * 100

print('KPIs by Product Category (sorted by Total Revenue):')
print('-' * 110)
print('{:<16} {:>14} {:>10} {:>10} {:>10} {:>10} {:>10} {:>14} {:>13}'.format(
    'Category', 'Revenue', 'Orders', 'Qty', 'AOV',
    'Avg Price', 'Avg Rtg', 'Avg Dlv(days)', 'Discount%'))
print('-' * 110)
for _, r in cat_kpi.iterrows():
    print('{:<16} {:>14,.0f} {:>10,} {:>10,} {:>10,.0f} {:>10,.0f} {:>10.2f} {:>14.2f} {:>12.1f}%'.format(
        r['Product_Category'], r['Total_Revenue'], int(r['Total_Orders']),
        int(r['Total_Qty']), r['Avg_Order_Value'], r['Avg_Unit_Price'],
        r['Avg_Rating'], r['Avg_Delivery_Days'], r['Discount_Pct_Orders']))

### 2.3 — KPIs by City

In [ ]:
# ── Revenue, Orders, AOV, Avg Rating, Avg Delivery by City ──────────────────
city_kpi = (
    df.groupby('City', observed=True)
    .agg(
        Total_Revenue    =('Total_Amount',       'sum'),
        Total_Orders     =('Order_ID',           'count'),
        Unique_Customers =('Customer_ID',        'nunique'),
        Avg_Order_Value  =('Total_Amount',       'mean'),
        Avg_Rating       =('Customer_Rating',    'mean'),
        Avg_Delivery_Days=('Delivery_Time_Days', 'mean'),
    )
    .sort_values('Total_Revenue', ascending=False)
    .reset_index()
)
city_kpi['Revenue_Per_Customer'] = city_kpi['Total_Revenue'] / city_kpi['Unique_Customers']
city_kpi['Revenue_Share_Pct']    = city_kpi['Total_Revenue'] / city_kpi['Total_Revenue'].sum() * 100

print('KPIs by City (sorted by Total Revenue):')
print('-' * 112)
print('{:<13} {:>14} {:>10} {:>13} {:>10} {:>10} {:>14} {:>14}'.format(
    'City', 'Revenue', 'Orders', 'Customers', 'AOV', 'Avg Rtg',
    'Avg Dlv(days)', 'Rev/Customer'))
print('-' * 112)
for _, r in city_kpi.iterrows():
    print('{:<13} {:>14,.0f} {:>10,} {:>13,} {:>10,.0f} {:>10.2f} {:>14.2f} {:>14,.0f}'.format(
        r['City'], r['Total_Revenue'], int(r['Total_Orders']),
        int(r['Unique_Customers']), r['Avg_Order_Value'],
        r['Avg_Rating'], r['Avg_Delivery_Days'], r['Revenue_Per_Customer']))

### 2.4 — Returning vs New Customer KPIs

In [ ]:
# ── KPIs split by Is_Returning_Customer ──────────────────────────────────────
ret_kpi = (
    df.groupby('Is_Returning_Customer', observed=True)
    .agg(
        Total_Revenue    =('Total_Amount',       'sum'),
        Total_Orders     =('Order_ID',           'count'),
        Unique_Customers =('Customer_ID',        'nunique'),
        Avg_Order_Value  =('Total_Amount',       'mean'),
        Avg_Rating       =('Customer_Rating',    'mean'),
        Avg_Delivery_Days=('Delivery_Time_Days', 'mean'),
        Discount_Pct_Orders=('Has_Discount',     'mean'),
    )
    .reset_index()
)
ret_kpi['Segment']             = ret_kpi['Is_Returning_Customer'].map({True: 'Returning', False: 'New'})
ret_kpi['Revenue_Share_Pct']   = ret_kpi['Total_Revenue'] / ret_kpi['Total_Revenue'].sum() * 100
ret_kpi['Order_Share_Pct']     = ret_kpi['Total_Orders']   / ret_kpi['Total_Orders'].sum()   * 100
ret_kpi['Discount_Pct_Orders'] = ret_kpi['Discount_Pct_Orders'] * 100

print('Returning vs New Customer KPIs:')
print('-' * 105)
print('{:<11} {:>13} {:>9} {:>11} {:>10} {:>10} {:>9} {:>13} {:>13}'.format(
    'Segment', 'Revenue', 'Orders', 'Customers', 'AOV', 'Avg Rtg',
    'Disc%', 'Rev Share%', 'Ord Share%'))
print('-' * 105)
for _, r in ret_kpi.iterrows():
    print('{:<11} {:>13,.0f} {:>9,} {:>11,} {:>10,.0f} {:>10.2f} {:>8.1f}% {:>12.1f}% {:>12.1f}%'.format(
        r['Segment'], r['Total_Revenue'], int(r['Total_Orders']),
        int(r['Unique_Customers']), r['Avg_Order_Value'],
        r['Avg_Rating'], r['Discount_Pct_Orders'],
        r['Revenue_Share_Pct'], r['Order_Share_Pct']))

### 2.5 — Monthly Revenue KPI Table

In [ ]:
# ── Monthly revenue, orders, AOV, MoM growth ─────────────────────────────────
monthly_kpi = (
    df.groupby('Year_Month', observed=True)
    .agg(
        Revenue=('Total_Amount', 'sum'),
        Orders =('Order_ID',     'count'),
    )
    .reset_index()
    .sort_values('Year_Month')
)
monthly_kpi['AOV']        = monthly_kpi['Revenue'] / monthly_kpi['Orders']
monthly_kpi['MoM_Growth%'] = monthly_kpi['Revenue'].pct_change() * 100

print('Monthly Revenue KPI Table:')
print('-' * 65)
print('{:<10} {:>13} {:>8} {:>10} {:>12}'.format(
    'Year-Month', 'Revenue', 'Orders', 'AOV', 'MoM Growth%'))
print('-' * 65)
for _, r in monthly_kpi.iterrows():
    mom = '{:>+.1f}%'.format(r['MoM_Growth%']) if not pd.isna(r['MoM_Growth%']) else '     —' 
    print('{:<10} {:>13,.0f} {:>8,} {:>10,.0f} {:>12}'.format(
        str(r['Year_Month']), r['Revenue'], int(r['Orders']), r['AOV'], mom))

### 2.6 — Store KPI Results for Later Sections

In [ ]:
# ── Store computed KPIs in a dictionary for reuse in later sections ───────────
KPI = {
    'total_revenue':           total_revenue,
    'total_orders':            total_orders,
    'unique_customers':        unique_customers,
    'aov':                     aov,
    'total_qty':               total_qty,
    'revenue_per_customer':    revenue_per_customer,
    'returning_rate':          returning_rate,
    'returning_orders':        int(returning_orders),
    'avg_delivery':            avg_delivery,
    'avg_rating':              avg_rating,
    'discount_penetration':    discount_penetration,
    'avg_discount_applied':    avg_discount_when_applied,
    'low_rating_rate':         low_rating_rate,
    'avg_orders_per_customer': avg_orders_per_customer,
    'hvo_rate':                hvo_rate,
    'p75_total_amount':        p75,
}

print('KPI dictionary stored. Keys:')
for k, v in KPI.items():
    print('  {:30s} {}'.format(k, round(float(v), 4)))
print()
print('Section 2 complete. Proceed to Section 3 — EDA Univariate.')

---
## Section 3 — EDA: Univariate Analysis

Each variable is examined individually.
- **Numerical variables**: histogram + KDE, boxplot, and summary statistics.
- **Categorical variables**: count plot with value labels.
- Charts are saved to `assets/` for use in the project report.
- Factual observations are printed below each chart, derived from computed values only.

### 3.0 — Rebuild Working DataFrame

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})
sns.set_palette('tab10')
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Has_Discount'] = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross) * 100, 0.0)
df['Age_Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,75],
                          labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
df['Year_Month'] = df['Date'].dt.to_period('M')
print('DataFrame ready: {:,} rows x {} columns'.format(df.shape[0], df.shape[1]))

### 3.1 — Numerical Variables: Distribution & Boxplot

In [ ]:
def plot_num(col, log_scale=False, bins=40, color='steelblue', unit=''):
    """Histogram+KDE and boxplot for a numerical column; save to assets/."""
    data = df[col].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histogram + KDE
    ax = axes[0]
    plot_data = np.log1p(data) if log_scale else data
    ax.hist(plot_data, bins=bins, color=color, edgecolor='white', alpha=0.85, density=True)
    plot_data.plot.kde(ax=ax, color='navy', linewidth=1.6)
    xlabel = 'log1p({})'.format(col) if log_scale else col
    ax.set_xlabel(xlabel + (' ({})'.format(unit) if unit else ''))
    ax.set_ylabel('Density')
    title_suffix = ' (log scale)' if log_scale else ''
    ax.set_title('{} - Distribution{}'.format(col, title_suffix))

    # Boxplot
    ax2 = axes[1]
    ax2.boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor=color, alpha=0.7),
                medianprops=dict(color='navy', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax2.set_ylabel(col + (' ({})'.format(unit) if unit else ''))
    ax2.set_title('{} - Boxplot'.format(col))
    ax2.set_xticks([])

    plt.tight_layout()
    fname = 'assets/eda_uni_{}.png'.format(col.lower())
    plt.savefig(fname, bbox_inches='tight')
    plt.show()

    # Factual stats
    q1, med, q3 = data.quantile([0.25, 0.50, 0.75])
    iqr = q3 - q1
    outliers = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    print('  Mean: {:,.2f}  |  Median: {:,.2f}  |  Std: {:,.2f}'.format(data.mean(), med, data.std()))
    print('  Min : {:,.2f}  |  Max: {:,.2f}'.format(data.min(), data.max()))
    print('  IQR : {:,.2f}  |  Outliers (IQR rule): {:,}'.format(iqr, outliers))
    print('  Skewness: {:.3f}'.format(data.skew()))
    print()

#### 3.1.1 — Total_Amount

In [ ]:
print('=== Total_Amount ===')
print('Formula: (Unit_Price x Quantity) - Discount_Amount  [derived column]')
plot_num('Total_Amount', log_scale=True, bins=50, color='steelblue')

#### 3.1.2 — Unit_Price

In [ ]:
print('=== Unit_Price ===')
plot_num('Unit_Price', log_scale=True, bins=50, color='darkorange')

#### 3.1.3 — Quantity

In [ ]:
print('=== Quantity ===')
fig, ax = plt.subplots(figsize=(7, 4))
qty_counts = df['Quantity'].value_counts().sort_index()
bars = ax.bar(qty_counts.index.astype(str), qty_counts.values,
              color='mediumseagreen', edgecolor='white')
for bar, v in zip(bars, qty_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
            '{:,}'.format(v), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Quantity')
ax.set_ylabel('Order Count')
ax.set_title('Quantity - Distribution (values 1 to 5)')
plt.tight_layout()
plt.savefig('assets/eda_uni_quantity.png', bbox_inches='tight')
plt.show()
print('  Value counts:')
print(df['Quantity'].value_counts().sort_index().to_string())
print('  Unique values: {}'.format(df['Quantity'].nunique()))
print('  Mean: {:.4f}  |  Std: {:.4f}'.format(df['Quantity'].mean(), df['Quantity'].std()))

#### 3.1.4 — Discount_Amount

In [ ]:
print('=== Discount_Amount ===')
zero_disc = (df['Discount_Amount'] == 0).sum()
nonzero   = (df['Discount_Amount'] > 0).sum()
print('  Orders with zero discount : {:,} ({:.1f}%)'.format(zero_disc, zero_disc/len(df)*100))
print('  Orders with discount      : {:,} ({:.1f}%)'.format(nonzero, nonzero/len(df)*100))
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.pie([zero_disc, nonzero],
       labels=['No Discount', 'Discount Applied'],
       autopct='%1.1f%%', startangle=90,
       colors=['#d3d3d3', 'coral'], wedgeprops={'edgecolor': 'white'})
ax.set_title('Discount_Amount - Zero vs Applied')

ax2 = axes[1]
disc_nonzero = df.loc[df['Discount_Amount'] > 0, 'Discount_Amount']
ax2.hist(np.log1p(disc_nonzero), bins=40, color='coral', edgecolor='white', alpha=0.85)
ax2.set_xlabel('log1p(Discount_Amount)')
ax2.set_ylabel('Count')
ax2.set_title('Discount_Amount - Distribution (non-zero, log scale)')

plt.tight_layout()
plt.savefig('assets/eda_uni_discount_amount.png', bbox_inches='tight')
plt.show()

disc_nonzero = df.loc[df['Discount_Amount'] > 0, 'Discount_Amount']
print('  Non-zero discount stats:')
print('    Mean  : {:,.2f}'.format(disc_nonzero.mean()))
print('    Median: {:,.2f}'.format(disc_nonzero.median()))
print('    Max   : {:,.2f}'.format(disc_nonzero.max()))
print('    Skew  : {:.3f}'.format(disc_nonzero.skew()))

#### 3.1.5 — Age

In [ ]:
print('=== Age ===')
plot_num('Age', log_scale=False, bins=30, color='mediumpurple')

#### 3.1.6 — Session_Duration_Minutes

In [ ]:
print('=== Session_Duration_Minutes ===')
plot_num('Session_Duration_Minutes', log_scale=False, bins=23, color='teal', unit='min')

#### 3.1.7 — Pages_Viewed

In [ ]:
print('=== Pages_Viewed ===')
plot_num('Pages_Viewed', log_scale=False, bins=18, color='goldenrod')

#### 3.1.8 — Delivery_Time_Days

In [ ]:
print('=== Delivery_Time_Days ===')
plot_num('Delivery_Time_Days', log_scale=False, bins=25, color='tomato', unit='days')

#### 3.1.9 — Customer_Rating

In [ ]:
print('=== Customer_Rating ===')
fig, ax = plt.subplots(figsize=(7, 4))
rating_counts = df['Customer_Rating'].value_counts().sort_index()
bars = ax.bar(rating_counts.index.astype(str), rating_counts.values,
              color=['#d9534f','#f0ad4e','#aaa','#5bc0de','#5cb85c'],
              edgecolor='white')
for bar, v in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            '{:,}'.format(v), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Customer Rating (1 = lowest, 5 = highest)')
ax.set_ylabel('Order Count')
ax.set_title('Customer_Rating - Distribution')
plt.tight_layout()
plt.savefig('assets/eda_uni_customer_rating.png', bbox_inches='tight')
plt.show()
print('  Value counts:')
print(df['Customer_Rating'].value_counts().sort_index().to_string())
print('  Mean  : {:.4f}'.format(df['Customer_Rating'].mean()))
print('  Median: {:.1f}'.format(df['Customer_Rating'].median()))
print('  Std   : {:.4f}'.format(df['Customer_Rating'].std()))

### 3.2 — Categorical Variables: Count Distributions

In [ ]:
def plot_cat(col, title=None, color='steelblue', horizontal=False, figsize=(8,4)):
    """Count plot for a categorical column with value labels; save to assets/."""
    counts = df[col].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=figsize)
    label = title or col
    if horizontal:
        bars = ax.barh(counts.index.astype(str), counts.values,
                       color=color, edgecolor='white', alpha=0.85)
        for bar, v in zip(bars, counts.values):
            ax.text(bar.get_width() + counts.values.max()*0.01,
                    bar.get_y() + bar.get_height()/2,
                    '{:,} ({:.1f}%)'.format(v, v/len(df)*100),
                    va='center', fontsize=9)
        ax.set_xlabel('Order Count')
        ax.set_ylabel(col)
        ax.invert_yaxis()
    else:
        bars = ax.bar(counts.index.astype(str), counts.values,
                      color=color, edgecolor='white', alpha=0.85)
        for bar, v in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.values.max()*0.01,
                    '{}\n({:.1f}%)'.format('{:,}'.format(v), v/len(df)*100),
                    ha='center', va='bottom', fontsize=8)
        ax.set_xlabel(col)
        ax.set_ylabel('Order Count')
    ax.set_title('{} - Order Count Distribution'.format(label))
    plt.tight_layout()
    fname = 'assets/eda_uni_{}.png'.format(col.lower())
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print('  {} value counts:'.format(col))
    for val, cnt in counts.items():
        print('    {:<25} {:>6,}  ({:.1f}%)'.format(str(val), cnt, cnt/len(df)*100))
    print()

#### 3.2.1 — Gender

In [ ]:
plot_cat('Gender', color='mediumpurple')

#### 3.2.2 — Product_Category

In [ ]:
plot_cat('Product_Category', horizontal=True, color='steelblue', figsize=(9,5))

#### 3.2.3 — Payment_Method

In [ ]:
plot_cat('Payment_Method', horizontal=True, color='darkorange', figsize=(9,4))

#### 3.2.4 — Device_Type

In [ ]:
plot_cat('Device_Type', color='teal')

#### 3.2.5 — Is_Returning_Customer

In [ ]:
counts = df['Is_Returning_Customer'].map({True: 'Returning', False: 'New'}).value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
bars = ax.bar(counts.index, counts.values,
              color=['steelblue','coral'], edgecolor='white', alpha=0.85)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            '{:,}\n({:.1f}%)'.format(v, v/len(df)*100),
            ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Customer Type')
ax.set_ylabel('Order Count')
ax.set_title('Returning vs New - Order Count')

ax2 = axes[1]
ax2.pie(counts.values, labels=counts.index,
        autopct='%1.1f%%', startangle=90,
        colors=['steelblue','coral'],
        wedgeprops={'edgecolor':'white', 'width': 0.55})
ax2.set_title('Returning vs New - Share')

plt.tight_layout()
plt.savefig('assets/eda_uni_returning_customer.png', bbox_inches='tight')
plt.show()
print('  Is_Returning_Customer value counts:')
for val, cnt in counts.items():
    print('    {:<12} {:>6,}  ({:.1f}%)'.format(val, cnt, cnt/len(df)*100))

### 3.3 — Section 3 Summary

In [ ]:
import os
saved = sorted([f for f in os.listdir('assets') if f.startswith('eda_uni_')])
print('Section 3 - EDA Univariate complete.')
print('Charts saved to assets/: {}'.format(len(saved)))
for f in saved:
    print('  assets/{}'.format(f))
print()
print('Proceed to Section 4 - EDA: Bivariate & Multivariate Analysis.')

---
## Section 4 — EDA: Bivariate & Multivariate Analysis

Relationships between pairs and groups of variables are examined here.
- Scatter plots, boxplots, grouped bar charts, and a correlation heatmap are used.
- Pearson correlation coefficients are computed where relevant.
- Charts are saved to `assets/`.
- Factual observations are printed from computed values only.

> **Note:** Correlation indicates association and does not establish causation.

### 4.0 — Rebuild Working DataFrame

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Has_Discount'] = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross) * 100, 0.0)
df['Age_Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,75],
                          labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
df['Year_Month'] = df['Date'].dt.to_period('M')
df['Customer_Type'] = df['Is_Returning_Customer'].map({True: 'Returning', False: 'New'})
print('DataFrame ready: {:,} rows x {} columns'.format(df.shape[0], df.shape[1]))

### 4.1 — Correlation Heatmap (Numerical Variables)

> Correlation indicates association and does not establish causation.

In [ ]:
num_cols = ['Age', 'Unit_Price', 'Quantity', 'Discount_Amount', 'Total_Amount',
            'Session_Duration_Minutes', 'Pages_Viewed', 'Delivery_Time_Days', 'Customer_Rating']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Numerical Variables')
plt.tight_layout()
plt.savefig('assets/eda_biv_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('Pearson correlations with Total_Amount (sorted by absolute value):')
ta_corr = corr['Total_Amount'].drop('Total_Amount').sort_values(key=abs, ascending=False)
for col, val in ta_corr.items():
    print('  {:<30} r = {:+.4f}'.format(col, val))
print()
print('Note: Correlation indicates association and does not establish causation.')

### 4.2 — Total_Amount vs Unit_Price

In [ ]:
r, p = stats.pearsonr(df['Unit_Price'], df['Total_Amount'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(np.log1p(df['Unit_Price']), np.log1p(df['Total_Amount']),
           alpha=0.15, s=8, color='steelblue')
m, b = np.polyfit(np.log1p(df['Unit_Price']), np.log1p(df['Total_Amount']), 1)
x_line = np.linspace(np.log1p(df['Unit_Price']).min(), np.log1p(df['Unit_Price']).max(), 200)
ax.plot(x_line, m*x_line + b, color='red', linewidth=1.5, label='Trend line')
ax.set_xlabel('log1p(Unit_Price)')
ax.set_ylabel('log1p(Total_Amount)')
ax.set_title('Total_Amount vs Unit_Price (log scale)  r = {:.4f}'.format(r))
ax.legend()
plt.tight_layout()
plt.savefig('assets/eda_biv_totalamount_vs_unitprice.png', bbox_inches='tight')
plt.show()
print('  Pearson r (Unit_Price vs Total_Amount): {:.4f}  p-value: {:.4e}'.format(r, p))
print('  Note: Correlation indicates association, not causation.')

### 4.3 — Total_Amount vs Quantity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
qty_groups = [df.loc[df['Quantity'] == q, 'Total_Amount'] for q in range(1, 6)]
ax.boxplot(qty_groups, tick_labels=['1','2','3','4','5'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6),
           medianprops=dict(color='navy', linewidth=2),
           flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_xlabel('Quantity')
ax.set_ylabel('Total_Amount')
ax.set_title('Total_Amount Distribution by Quantity')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: '{:,.0f}'.format(x)))
plt.tight_layout()
plt.savefig('assets/eda_biv_totalamount_by_quantity.png', bbox_inches='tight')
plt.show()
r_qty, p_qty = stats.pearsonr(df['Quantity'], df['Total_Amount'])
print('  Mean Total_Amount by Quantity:')
for q in range(1, 6):
    m = df.loc[df['Quantity'] == q, 'Total_Amount'].mean()
    print('    Qty {}: {:,.2f}'.format(q, m))
print('  Pearson r (Quantity vs Total_Amount): {:.4f}  p-value: {:.4e}'.format(r_qty, p_qty))

### 4.4 — Total_Amount vs Discount_Amount (discounted orders only)

In [ ]:
df_disc = df[df['Has_Discount']].copy()
r_d, p_d = stats.pearsonr(df_disc['Discount_Amount'], df_disc['Total_Amount'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(np.log1p(df_disc['Discount_Amount']), np.log1p(df_disc['Total_Amount']),
           alpha=0.2, s=8, color='coral')
m2, b2 = np.polyfit(np.log1p(df_disc['Discount_Amount']), np.log1p(df_disc['Total_Amount']), 1)
x2 = np.linspace(np.log1p(df_disc['Discount_Amount']).min(),
                 np.log1p(df_disc['Discount_Amount']).max(), 200)
ax.plot(x2, m2*x2 + b2, color='darkred', linewidth=1.5, label='Trend line')
ax.set_xlabel('log1p(Discount_Amount)')
ax.set_ylabel('log1p(Total_Amount)')
ax.set_title('Total_Amount vs Discount_Amount (discounted orders, log scale)  r = {:.4f}'.format(r_d))
ax.legend()
plt.tight_layout()
plt.savefig('assets/eda_biv_totalamount_vs_discount.png', bbox_inches='tight')
plt.show()
print('  Discounted orders: {:,}'.format(len(df_disc)))
print('  Pearson r (Discount_Amount vs Total_Amount, discounted only): {:.4f}  p: {:.4e}'.format(r_d, p_d))
print('  Note: Higher discount amounts tend to co-occur with higher-priced orders by construction')
print('        (Total_Amount = Unit_Price*Qty - Discount_Amount). Interpret with care.')

### 4.5 — Total_Amount vs Session_Duration_Minutes

In [ ]:
r_s, p_s = stats.pearsonr(df['Session_Duration_Minutes'], df['Total_Amount'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Session_Duration_Minutes'], np.log1p(df['Total_Amount']),
           alpha=0.12, s=8, color='teal')
m3, b3 = np.polyfit(df['Session_Duration_Minutes'], np.log1p(df['Total_Amount']), 1)
x3 = np.linspace(df['Session_Duration_Minutes'].min(), df['Session_Duration_Minutes'].max(), 200)
ax.plot(x3, m3*x3 + b3, color='darkred', linewidth=1.5, label='Trend line')
ax.set_xlabel('Session_Duration_Minutes')
ax.set_ylabel('log1p(Total_Amount)')
ax.set_title('Total_Amount vs Session_Duration_Minutes  r = {:.4f}'.format(r_s))
ax.legend()
plt.tight_layout()
plt.savefig('assets/eda_biv_totalamount_vs_session.png', bbox_inches='tight')
plt.show()
print('  Mean Total_Amount by Session_Duration_Minutes (selected):')
for dur in sorted(df['Session_Duration_Minutes'].unique())[:5]:
    m = df.loc[df['Session_Duration_Minutes'] == dur, 'Total_Amount'].mean()
    print('    {} min: {:,.2f}'.format(dur, m))
print('  ...')
print('  Pearson r (Session_Duration_Minutes vs Total_Amount): {:.4f}  p: {:.4e}'.format(r_s, p_s))

### 4.6 — Total_Amount vs Pages_Viewed

In [ ]:
r_pv, p_pv = stats.pearsonr(df['Pages_Viewed'], df['Total_Amount'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Pages_Viewed'], np.log1p(df['Total_Amount']),
           alpha=0.12, s=8, color='goldenrod')
m4, b4 = np.polyfit(df['Pages_Viewed'], np.log1p(df['Total_Amount']), 1)
x4 = np.linspace(df['Pages_Viewed'].min(), df['Pages_Viewed'].max(), 200)
ax.plot(x4, m4*x4 + b4, color='darkred', linewidth=1.5, label='Trend line')
ax.set_xlabel('Pages_Viewed')
ax.set_ylabel('log1p(Total_Amount)')
ax.set_title('Total_Amount vs Pages_Viewed  r = {:.4f}'.format(r_pv))
ax.legend()
plt.tight_layout()
plt.savefig('assets/eda_biv_totalamount_vs_pages.png', bbox_inches='tight')
plt.show()
print('  Pearson r (Pages_Viewed vs Total_Amount): {:.4f}  p: {:.4e}'.format(r_pv, p_pv))

### 4.7 — Delivery_Time_Days vs Customer_Rating

In [ ]:
r_dr, p_dr = stats.pearsonr(df['Delivery_Time_Days'], df['Customer_Rating'])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(df['Delivery_Time_Days'], df['Customer_Rating'],
                alpha=0.08, s=8, color='tomato')
m5, b5 = np.polyfit(df['Delivery_Time_Days'], df['Customer_Rating'], 1)
x5 = np.linspace(df['Delivery_Time_Days'].min(), df['Delivery_Time_Days'].max(), 200)
axes[0].plot(x5, m5*x5 + b5, color='darkred', linewidth=1.5, label='Trend line')
axes[0].set_xlabel('Delivery_Time_Days')
axes[0].set_ylabel('Customer_Rating')
axes[0].set_title('Delivery Time vs Rating  r = {:.4f}'.format(r_dr))
axes[0].legend()

# Mean rating by delivery bucket
bins = [0, 3, 6, 10, 25]
labels = ['1-3d', '4-6d', '7-10d', '>10d']
df['Delivery_Bucket'] = pd.cut(df['Delivery_Time_Days'], bins=bins, labels=labels)
bucket_rating = df.groupby('Delivery_Bucket', observed=True)['Customer_Rating'].mean()
axes[1].bar(bucket_rating.index.astype(str), bucket_rating.values,
            color=['#5cb85c','#5bc0de','#f0ad4e','#d9534f'], edgecolor='white')
for i, (lbl, val) in enumerate(bucket_rating.items()):
    axes[1].text(i, val + 0.02, '{:.2f}'.format(val), ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Delivery Time Bucket')
axes[1].set_ylabel('Mean Customer Rating')
axes[1].set_title('Avg Rating by Delivery Time Bucket')
axes[1].set_ylim(0, 5.3)

plt.tight_layout()
plt.savefig('assets/eda_biv_delivery_vs_rating.png', bbox_inches='tight')
plt.show()

print('  Pearson r (Delivery_Time_Days vs Customer_Rating): {:.4f}  p: {:.4e}'.format(r_dr, p_dr))
print('  Mean Customer_Rating by delivery bucket:')
for lbl, val in bucket_rating.items():
    print('    {:<8} {:.4f}'.format(str(lbl), val))
print('  Note: Correlation indicates association and does not establish causation.')

### 4.8 — Revenue by Product_Category and Gender

In [ ]:
cat_gender = (
    df.groupby(['Product_Category', 'Gender'], observed=True)['Total_Amount']
    .sum().reset_index()
    .pivot(index='Product_Category', columns='Gender', values='Total_Amount')
    .fillna(0)
)
cat_gender = cat_gender.loc[cat_gender.sum(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(cat_gender.index))
width = 0.28
colors = ['#5b9bd5','#ed7d31','#70ad47']
for i, col in enumerate(cat_gender.columns):
    bars = ax.bar(x + i*width, cat_gender[col], width, label=col,
                  color=colors[i], edgecolor='white', alpha=0.87)
ax.set_xticks(x + width)
ax.set_xticklabels(cat_gender.index, rotation=25, ha='right')
ax.set_ylabel('Total Revenue')
ax.set_title('Revenue by Product Category and Gender')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: '{:,.0f}'.format(x)))
ax.legend(title='Gender')
plt.tight_layout()
plt.savefig('assets/eda_biv_revenue_category_gender.png', bbox_inches='tight')
plt.show()
print('  Revenue by Category & Gender (top category per gender):')
for col in cat_gender.columns:
    top = cat_gender[col].idxmax()
    print('    {:<8}: top category = {} ({:,.0f})'.format(col, top, cat_gender.loc[top, col]))

### 4.9 — Revenue and AOV by Device_Type

In [ ]:
dev = df.groupby('Device_Type', observed=True).agg(
    Total_Revenue=('Total_Amount','sum'),
    Total_Orders=('Order_ID','count'),
    AOV=('Total_Amount','mean')
).reset_index().sort_values('Total_Revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#5b9bd5','#ed7d31','#70ad47']
bars1 = axes[0].bar(dev['Device_Type'], dev['Total_Revenue'],
                    color=colors[:len(dev)], edgecolor='white')
for bar, v in zip(bars1, dev['Total_Revenue']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50000,
                 '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Total Revenue by Device Type')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

bars2 = axes[1].bar(dev['Device_Type'], dev['AOV'],
                    color=colors[:len(dev)], edgecolor='white')
for bar, v in zip(bars2, dev['AOV']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Average Order Value')
axes[1].set_title('AOV by Device Type')

plt.tight_layout()
plt.savefig('assets/eda_biv_revenue_by_device.png', bbox_inches='tight')
plt.show()
print('  Device Type summary:')
print(dev.to_string(index=False))

### 4.10 — AOV by Payment_Method

In [ ]:
pay = df.groupby('Payment_Method', observed=True).agg(
    Total_Revenue=('Total_Amount','sum'),
    Orders=('Order_ID','count'),
    AOV=('Total_Amount','mean')
).reset_index().sort_values('AOV', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(pay['Payment_Method'], pay['AOV'],
               color='steelblue', edgecolor='white', alpha=0.87)
for bar, v in zip(bars, pay['AOV']):
    ax.text(bar.get_width() + pay['AOV'].max()*0.01, bar.get_y()+bar.get_height()/2,
            '{:,.0f}'.format(v), va='center', fontsize=9)
ax.set_xlabel('Average Order Value')
ax.set_title('AOV by Payment Method')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('assets/eda_biv_aov_by_payment.png', bbox_inches='tight')
plt.show()
print('  AOV by Payment Method:')
print(pay[['Payment_Method','AOV','Orders']].to_string(index=False))

### 4.11 — Customer_Rating by Product_Category

In [ ]:
cat_rating = df.groupby('Product_Category', observed=True)['Customer_Rating'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(cat_rating.index, cat_rating.values,
               color='steelblue', edgecolor='white', alpha=0.87)
for bar, v in zip(bars, cat_rating.values):
    ax.text(bar.get_width() + 0.01, bar.get_y()+bar.get_height()/2,
            '{:.2f}'.format(v), va='center', fontsize=9)
ax.set_xlabel('Average Customer Rating')
ax.set_title('Average Customer Rating by Product Category')
ax.set_xlim(0, 5.3)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('assets/eda_biv_rating_by_category.png', bbox_inches='tight')
plt.show()
print('  Average Customer_Rating by Product_Category:')
for cat, val in cat_rating.items():
    print('    {:<16} {:.4f}'.format(cat, val))
overall_avg = df['Customer_Rating'].mean()
print('  Overall average: {:.4f}'.format(overall_avg))

### 4.12 — Delivery_Time_Days by Product_Category

In [ ]:
cat_del = df.groupby('Product_Category', observed=True)['Delivery_Time_Days'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(cat_del.index, cat_del.values,
               color='tomato', edgecolor='white', alpha=0.87)
for bar, v in zip(bars, cat_del.values):
    ax.text(bar.get_width() + 0.05, bar.get_y()+bar.get_height()/2,
            '{:.2f} d'.format(v), va='center', fontsize=9)
ax.set_xlabel('Average Delivery Time (days)')
ax.set_title('Average Delivery Time by Product Category')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('assets/eda_biv_delivery_by_category.png', bbox_inches='tight')
plt.show()
print('  Average Delivery_Time_Days by Product_Category:')
for cat, val in cat_del.items():
    print('    {:<16} {:.4f} days'.format(cat, val))
print('  Overall average: {:.4f} days'.format(df['Delivery_Time_Days'].mean()))

### 4.13 — Returning vs New Customers: Revenue by Device Type and Category

In [ ]:
# By Device_Type
ret_dev = df.groupby(['Device_Type', 'Customer_Type'], observed=True)['Total_Amount'].sum().reset_index()
ret_dev_pivot = ret_dev.pivot(index='Device_Type', columns='Customer_Type', values='Total_Amount').fillna(0)

# By Product_Category
ret_cat = df.groupby(['Product_Category', 'Customer_Type'], observed=True)['Total_Amount'].sum().reset_index()
ret_cat_pivot = ret_cat.pivot(index='Product_Category', columns='Customer_Type', values='Total_Amount').fillna(0)
ret_cat_pivot = ret_cat_pivot.loc[ret_cat_pivot.sum(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Device
x1 = np.arange(len(ret_dev_pivot.index))
w = 0.35
cols_present = ret_dev_pivot.columns.tolist()
clr = {'Returning': 'steelblue', 'New': 'coral'}
for i, col in enumerate(cols_present):
    axes[0].bar(x1 + i*w, ret_dev_pivot[col], w, label=col,
                color=clr.get(col,'grey'), edgecolor='white', alpha=0.87)
axes[0].set_xticks(x1 + w/2)
axes[0].set_xticklabels(ret_dev_pivot.index)
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Revenue by Device Type: Returning vs New')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
axes[0].legend()

# Category
x2 = np.arange(len(ret_cat_pivot.index))
for i, col in enumerate(cols_present):
    axes[1].bar(x2 + i*w, ret_cat_pivot[col], w, label=col,
                color=clr.get(col,'grey'), edgecolor='white', alpha=0.87)
axes[1].set_xticks(x2 + w/2)
axes[1].set_xticklabels(ret_cat_pivot.index, rotation=30, ha='right')
axes[1].set_ylabel('Total Revenue')
axes[1].set_title('Revenue by Category: Returning vs New')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/eda_biv_returning_vs_new_revenue.png', bbox_inches='tight')
plt.show()

ret_aov = df.groupby('Customer_Type', observed=True)['Total_Amount'].mean()
print('  AOV by Customer Type:')
for ct, val in ret_aov.items():
    print('    {:<12} {:,.2f}'.format(ct, val))

### 4.14 — Section 4 Summary

In [ ]:
import os
saved = sorted([f for f in os.listdir('assets') if f.startswith('eda_biv_')])
print('Section 4 - EDA Bivariate & Multivariate complete.')
print('Charts saved: {}'.format(len(saved)))
for f in saved:
    print('  assets/{}'.format(f))
print()
print('Proceed to Section 5 - Time-Series & Sales Trend Analysis.')

---
## Section 5 — Time-Series & Sales Trend Analysis

Sales trends are analysed across monthly, quarterly, and yearly dimensions.
- All time series are sorted chronologically.
- MoM growth for the first month is unavailable (NaN) by definition — not replaced.
- Charts are saved to `assets/`.
- Observations are printed from computed values only.

### 5.0 — Rebuild Working DataFrame

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 10,
})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter']    = df['Date'].dt.quarter
df['Year_Month'] = df['Date'].dt.to_period('M')
df['Year_Q']     = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)
df['Has_Discount'] = df['Discount_Amount'] > 0
df['Customer_Type'] = df['Is_Returning_Customer'].map({True: 'Returning', False: 'New'})
print('DataFrame ready: {:,} rows x {} columns'.format(df.shape[0], df.shape[1]))

### 5.1 — Monthly Aggregates

In [ ]:
monthly = (
    df.groupby('Year_Month', observed=True)
    .agg(
        Revenue  = ('Total_Amount', 'sum'),
        Orders   = ('Order_ID',     'count'),
        Qty      = ('Quantity',     'sum'),
    )
    .sort_index()
    .reset_index()
)
monthly['AOV']          = monthly['Revenue'] / monthly['Orders']
monthly['MoM_Growth_Pct'] = monthly['Revenue'].pct_change() * 100
monthly['Period_Str']   = monthly['Year_Month'].astype(str)

print('Monthly aggregates computed: {} periods'.format(len(monthly)))
print(monthly[['Period_Str','Revenue','Orders','Qty','AOV','MoM_Growth_Pct']].to_string(index=False))

### 5.2 — Monthly Revenue Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['Period_Str'], monthly['Revenue'],
        marker='o', markersize=5, linewidth=2, color='steelblue', label='Monthly Revenue')
ax.fill_between(monthly['Period_Str'], monthly['Revenue'], alpha=0.12, color='steelblue')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Total Revenue')
ax.set_title('Monthly Revenue Trend (Jan 2023 - Mar 2024)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Period_Str'], rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('assets/ts_monthly_revenue.png', bbox_inches='tight')
plt.show()

peak = monthly.loc[monthly['Revenue'].idxmax()]
trough = monthly.loc[monthly['Revenue'].idxmin()]
print('  Peak revenue month  : {} ({:,.0f})'.format(peak['Period_Str'], peak['Revenue']))
print('  Lowest revenue month: {} ({:,.0f})'.format(trough['Period_Str'], trough['Revenue']))
print('  Total months in dataset: {}'.format(len(monthly)))

### 5.3 — Monthly Order Count Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['Period_Str'], monthly['Orders'],
        marker='s', markersize=4, linewidth=2, color='darkorange')
ax.fill_between(monthly['Period_Str'], monthly['Orders'], alpha=0.12, color='darkorange')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Order Count')
ax.set_title('Monthly Order Count Trend')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Period_Str'], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('assets/ts_monthly_orders.png', bbox_inches='tight')
plt.show()

peak_o = monthly.loc[monthly['Orders'].idxmax()]
print('  Peak orders month: {} ({:,})'.format(peak_o['Period_Str'], int(peak_o['Orders'])))

### 5.4 — Monthly Average Order Value Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['Period_Str'], monthly['AOV'],
        marker='^', markersize=5, linewidth=2, color='mediumseagreen')
overall_aov = df['Total_Amount'].mean()
ax.axhline(overall_aov, color='grey', linestyle='--', linewidth=1,
           label='Overall AOV ({:,.0f})'.format(overall_aov))
ax.set_xlabel('Year-Month')
ax.set_ylabel('Average Order Value')
ax.set_title('Monthly AOV Trend')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Period_Str'], rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('assets/ts_monthly_aov.png', bbox_inches='tight')
plt.show()

print('  Monthly AOV range: {:.0f} - {:.0f}'.format(monthly['AOV'].min(), monthly['AOV'].max()))
print('  Overall AOV: {:.2f}'.format(overall_aov))

### 5.5 — Month-over-Month Revenue Growth

In [ ]:
mom = monthly.dropna(subset=['MoM_Growth_Pct']).copy()
colors_mom = ['#5cb85c' if v >= 0 else '#d9534f' for v in mom['MoM_Growth_Pct']]

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(mom['Period_Str'], mom['MoM_Growth_Pct'], color=colors_mom, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year-Month')
ax.set_ylabel('MoM Revenue Growth (%)')
ax.set_title('Month-over-Month Revenue Growth Rate\n(first month excluded - MoM unavailable)')
ax.set_xticks(range(len(mom)))
ax.set_xticklabels(mom['Period_Str'], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('assets/ts_mom_growth.png', bbox_inches='tight')
plt.show()

pos = (mom['MoM_Growth_Pct'] > 0).sum()
neg = (mom['MoM_Growth_Pct'] <= 0).sum()
print('  MoM growth available for {} months (first month excluded as NaN)'.format(len(mom)))
print('  Positive growth months: {}'.format(pos))
print('  Negative/flat months  : {}'.format(neg))
print('  Max MoM growth : {:.2f}% ({})'.format(mom['MoM_Growth_Pct'].max(), mom.loc[mom['MoM_Growth_Pct'].idxmax(), 'Period_Str']))
print('  Min MoM growth : {:.2f}% ({})'.format(mom['MoM_Growth_Pct'].min(), mom.loc[mom['MoM_Growth_Pct'].idxmin(), 'Period_Str']))

### 5.6 — Quarterly Revenue Comparison

In [ ]:
quarterly = (
    df.groupby('Year_Q', observed=True)
    .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'))
    .reset_index()
    .sort_values('Year_Q')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bar_colors = sns.color_palette('tab10', len(quarterly))
bars = axes[0].bar(quarterly['Year_Q'], quarterly['Revenue'],
                   color=bar_colors, edgecolor='white')
for bar, v in zip(bars, quarterly['Revenue']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+15000,
                 '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=8, rotation=15)
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Quarterly Revenue')
axes[0].set_xticklabels(quarterly['Year_Q'], rotation=30, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

bars2 = axes[1].bar(quarterly['Year_Q'], quarterly['Orders'],
                    color=bar_colors, edgecolor='white')
for bar, v in zip(bars2, quarterly['Orders']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 '{:,}'.format(int(v)), ha='center', va='bottom', fontsize=8, rotation=15)
axes[1].set_ylabel('Order Count')
axes[1].set_title('Quarterly Order Count')
axes[1].set_xticklabels(quarterly['Year_Q'], rotation=30, ha='right')

plt.tight_layout()
plt.savefig('assets/ts_quarterly.png', bbox_inches='tight')
plt.show()

print('  Quarterly summary:')
print(quarterly.to_string(index=False))

### 5.7 — Revenue and Orders by Year

In [ ]:
yearly = (
    df.groupby('Year', observed=True)
    .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'),
         AOV=('Total_Amount','mean'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(yearly['Year'].astype(str), yearly['Revenue'],
              color=['steelblue','darkorange'], edgecolor='white')
for bar, v in zip(bars, yearly['Revenue']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20000,
            '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Total Revenue')
ax.set_title('Total Revenue by Year\n(2024 is partial: Jan-Mar only)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
plt.tight_layout()
plt.savefig('assets/ts_yearly_revenue.png', bbox_inches='tight')
plt.show()

print('  NOTE: 2024 data covers Jan 1 - Mar 25 only (partial year).')
print(yearly.to_string(index=False))

### 5.8 — Product Category Revenue Trends Over Time

In [ ]:
cat_monthly = (
    df.groupby(['Year_Month', 'Product_Category'], observed=True)['Total_Amount']
    .sum().reset_index().sort_values('Year_Month')
)
cat_monthly['Period_Str'] = cat_monthly['Year_Month'].astype(str)
categories = sorted(df['Product_Category'].unique())
palette = sns.color_palette('tab10', len(categories))

fig, ax = plt.subplots(figsize=(14, 6))
for cat, color in zip(categories, palette):
    subset = cat_monthly[cat_monthly['Product_Category'] == cat]
    ax.plot(subset['Period_Str'], subset['Total_Amount'],
            marker='o', markersize=3, linewidth=1.4, label=cat, color=color)
ax.set_xlabel('Year-Month')
ax.set_ylabel('Total Revenue')
ax.set_title('Monthly Revenue by Product Category')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ticks = cat_monthly['Period_Str'].unique()
ax.set_xticks(range(len(ticks)))
ax.set_xticklabels(ticks, rotation=45, ha='right')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('assets/ts_category_monthly_trend.png', bbox_inches='tight')
plt.show()

print('  Categories plotted: {}'.format(len(categories)))

### 5.9 — Returning vs New Customer Revenue Trends

In [ ]:
ret_monthly = (
    df.groupby(['Year_Month', 'Customer_Type'], observed=True)['Total_Amount']
    .sum().reset_index().sort_values('Year_Month')
)
ret_monthly['Period_Str'] = ret_monthly['Year_Month'].astype(str)

fig, ax = plt.subplots(figsize=(13, 4))
clr = {'Returning': 'steelblue', 'New': 'coral'}
for ctype in ['Returning', 'New']:
    subset = ret_monthly[ret_monthly['Customer_Type'] == ctype]
    ax.plot(subset['Period_Str'], subset['Total_Amount'],
            marker='o', markersize=4, linewidth=2,
            color=clr[ctype], label=ctype)
ax.set_xlabel('Year-Month')
ax.set_ylabel('Total Revenue')
ax.set_title('Monthly Revenue: Returning vs New Customers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ax.set_xticks(range(len(ret_monthly['Period_Str'].unique())))
ax.set_xticklabels(ret_monthly['Period_Str'].unique(), rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('assets/ts_returning_vs_new_trend.png', bbox_inches='tight')
plt.show()

print('  Overall revenue split:')
split = df.groupby('Customer_Type', observed=True)['Total_Amount'].sum()
for ct, val in split.items():
    print('    {:<12} {:,.2f}  ({:.1f}%)'.format(ct, val, val/split.sum()*100))

### 5.10 — Monthly Quantity Sold Trend

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(monthly['Period_Str'], monthly['Qty'],
       color='mediumseagreen', edgecolor='white', alpha=0.85)
ax.plot(monthly['Period_Str'], monthly['Qty'],
        color='darkgreen', linewidth=1.5, marker='o', markersize=4)
ax.set_xlabel('Year-Month')
ax.set_ylabel('Total Quantity Sold')
ax.set_title('Monthly Quantity Sold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Period_Str'], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('assets/ts_monthly_qty.png', bbox_inches='tight')
plt.show()

print('  Total quantity sold: {:,}'.format(int(monthly['Qty'].sum())))
print('  Peak quantity month: {} ({:,})'.format(
    monthly.loc[monthly['Qty'].idxmax(), 'Period_Str'],
    int(monthly['Qty'].max())))

### 5.11 — Section 5 Summary

In [ ]:
import os
saved = sorted([f for f in os.listdir('assets') if f.startswith('ts_')])
print('Section 5 - Time-Series & Sales Trend Analysis complete.')
print('Charts saved: {}'.format(len(saved)))
for f in saved:
    print('  assets/{}'.format(f))
print()
print('Proceed to Section 6 - Customer Behaviour Analysis.')

---
## Section 6 — Customer Behaviour Analysis

This section analyses behaviour patterns at both the **order level** and the **customer level**.
- Order-level: each row = one order.
- Customer-level: aggregated by `Customer_ID` (one row per customer).
- All values are computed from the actual dataset.
- Charts are saved to `assets/`.

### 6.0 — Rebuild Working DataFrame

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Has_Discount'] = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross) * 100, 0.0)
df['Age_Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,75],
                          labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
df['Customer_Type'] = df['Is_Returning_Customer'].map({True: 'Returning', False: 'New'})

# ── Customer-level aggregates (one row per Customer_ID) ──────────────────
cust = (
    df.groupby('Customer_ID')
    .agg(
        Order_Count        = ('Order_ID',              'count'),
        Total_Spend        = ('Total_Amount',          'sum'),
        Avg_Order_Value    = ('Total_Amount',          'mean'),
        Avg_Rating         = ('Customer_Rating',       'mean'),
        Avg_Delivery       = ('Delivery_Time_Days',    'mean'),
        Is_Returning       = ('Is_Returning_Customer', 'max'),
        Gender             = ('Gender',                'first'),
        City               = ('City',                  'first'),
        Age                = ('Age',                   'first'),
        Device_Type        = ('Device_Type',           'first'),
        Payment_Method     = ('Payment_Method',        'first'),
    )
    .reset_index()
)
cust['Age_Group'] = pd.cut(cust['Age'], bins=[17,25,35,45,55,75],
                            labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
cust['Customer_Type'] = cust['Is_Returning'].map({True: 'Returning', False: 'New'})

print('Order-level df  : {:,} rows'.format(len(df)))
print('Customer-level  : {:,} unique customers'.format(len(cust)))
print('Cust columns    :', cust.columns.tolist())

### 6.1 — Customer Order Frequency Distribution

In [ ]:
freq = cust['Order_Count'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
axes[0].bar(freq.index.astype(str), freq.values, color='steelblue', edgecolor='white')
for i, (idx, v) in enumerate(freq.items()):
    axes[0].text(i, v + 5, '{:,}'.format(v), ha='center', va='bottom', fontsize=8)
axes[0].set_xlabel('Orders per Customer')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Customer Order Frequency Distribution')

# Cumulative distribution
cum = freq.cumsum() / freq.sum() * 100
axes[1].plot(freq.index, cum.values, marker='o', color='darkorange', linewidth=2)
axes[1].axhline(80, color='grey', linestyle='--', linewidth=1, label='80% line')
axes[1].set_xlabel('Orders per Customer')
axes[1].set_ylabel('Cumulative % of Customers')
axes[1].set_title('Cumulative Customer Order Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/cb_order_frequency.png', bbox_inches='tight')
plt.show()

print('  Orders per customer distribution:')
for k, v in freq.items():
    print('    {:>3} orders: {:>5,} customers ({:.1f}%)'.format(k, v, v/len(cust)*100))
print('  Mean  : {:.2f}'.format(cust['Order_Count'].mean()))
print('  Median: {:.1f}'.format(cust['Order_Count'].median()))
print('  Max   : {}'.format(cust['Order_Count'].max()))

### 6.2 — Customer Total Spend Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(np.log1p(cust['Total_Spend']), bins=40, color='teal', edgecolor='white', alpha=0.85, density=True)
np.log1p(cust['Total_Spend']).plot.kde(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_xlabel('log1p(Total_Spend per Customer)')
axes[0].set_ylabel('Density')
axes[0].set_title('Customer Total Spend Distribution (log scale)')

axes[1].boxplot(cust['Total_Spend'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='teal', alpha=0.6),
                medianprops=dict(color='navy', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_ylabel('Total_Spend per Customer')
axes[1].set_title('Customer Total Spend Boxplot')
axes[1].set_xticks([])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

plt.tight_layout()
plt.savefig('assets/cb_customer_spend_dist.png', bbox_inches='tight')
plt.show()

q1, med, q3 = cust['Total_Spend'].quantile([0.25, 0.50, 0.75])
print('  Customer Total Spend (customer-level):')
print('    Mean  : {:,.2f}'.format(cust['Total_Spend'].mean()))
print('    Median: {:,.2f}'.format(med))
print('    Q1    : {:,.2f}  Q3: {:,.2f}'.format(q1, q3))
print('    Max   : {:,.2f}'.format(cust['Total_Spend'].max()))
print('    Skew  : {:.3f}'.format(cust['Total_Spend'].skew()))

### 6.3 — Top 15 Customers by Total Spend

In [ ]:
top15 = cust.nlargest(15, 'Total_Spend')[['Customer_ID','Total_Spend','Order_Count','Avg_Order_Value','Gender','City']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top15['Customer_ID'], top15['Total_Spend'],
               color='steelblue', edgecolor='white', alpha=0.87)
for bar, v in zip(bars, top15['Total_Spend']):
    ax.text(bar.get_width() + top15['Total_Spend'].max()*0.01,
            bar.get_y() + bar.get_height()/2,
            '{:,.0f}'.format(v), va='center', fontsize=8)
ax.set_xlabel('Total Spend')
ax.set_title('Top 15 Customers by Total Spend')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))
plt.tight_layout()
plt.savefig('assets/cb_top_customers.png', bbox_inches='tight')
plt.show()

print('  Top 15 customers by total spend:')
print(top15.to_string(index=False))

### 6.4 — Age-Group Revenue and Order Count (Order Level)

In [ ]:
age_rev = (
    df.groupby('Age_Group', observed=True)
    .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'),
         AOV=('Total_Amount','mean'), Avg_Rating=('Customer_Rating','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = sns.color_palette('tab10', len(age_rev))
bars1 = axes[0].bar(age_rev['Age_Group'].astype(str), age_rev['Revenue'],
                    color=colors, edgecolor='white')
for bar, v in zip(bars1, age_rev['Revenue']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20000,
                 '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Revenue by Age Group')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

bars2 = axes[1].bar(age_rev['Age_Group'].astype(str), age_rev['Orders'],
                    color=colors, edgecolor='white')
for bar, v in zip(bars2, age_rev['Orders']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 '{:,}'.format(int(v)), ha='center', va='bottom', fontsize=8)
axes[1].set_ylabel('Order Count')
axes[1].set_title('Orders by Age Group')

plt.tight_layout()
plt.savefig('assets/cb_age_group_revenue.png', bbox_inches='tight')
plt.show()

print('  Revenue and orders by Age Group:')
print(age_rev.to_string(index=False))

### 6.5 — Gender-Wise Revenue and Order Count

In [ ]:
gender_rev = (
    df.groupby('Gender', observed=True)
    .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'),
         AOV=('Total_Amount','mean'), Avg_Rating=('Customer_Rating','mean'))
    .reset_index().sort_values('Revenue', ascending=False)
)
gender_rev['Revenue_Share'] = gender_rev['Revenue'] / gender_rev['Revenue'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
clrs = ['#5b9bd5','#ed7d31','#70ad47']
axes[0].bar(gender_rev['Gender'], gender_rev['Revenue'], color=clrs, edgecolor='white')
for i, v in enumerate(gender_rev['Revenue']):
    axes[0].text(i, v+10000, '{:,.0f}\n({:.1f}%)'.format(v, gender_rev['Revenue_Share'].iloc[i]),
                 ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Revenue by Gender')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

axes[1].bar(gender_rev['Gender'], gender_rev['AOV'], color=clrs, edgecolor='white')
for i, v in enumerate(gender_rev['AOV']):
    axes[1].text(i, v+5, '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Average Order Value')
axes[1].set_title('AOV by Gender')

plt.tight_layout()
plt.savefig('assets/cb_gender_revenue.png', bbox_inches='tight')
plt.show()

print('  Gender summary:')
print(gender_rev.to_string(index=False))

### 6.6 — Device-Wise Behaviour

In [ ]:
dev_beh = (
    df.groupby('Device_Type', observed=True)
    .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'),
         AOV=('Total_Amount','mean'), Avg_Session=('Session_Duration_Minutes','mean'),
         Avg_Pages=('Pages_Viewed','mean'), Avg_Rating=('Customer_Rating','mean'))
    .reset_index().sort_values('Revenue', ascending=False)
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
clrs = ['#5b9bd5','#ed7d31','#70ad47']
for ax, col, title in zip(axes,
    ['Revenue','Avg_Session','Avg_Pages'],
    ['Revenue by Device','Avg Session Duration (min)','Avg Pages Viewed']):
    bars = ax.bar(dev_beh['Device_Type'], dev_beh[col], color=clrs, edgecolor='white')
    for bar, v in zip(bars, dev_beh[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                '{:,.1f}'.format(v), ha='center', va='bottom', fontsize=9)
    ax.set_title(title)
    if col == 'Revenue':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

plt.tight_layout()
plt.savefig('assets/cb_device_behaviour.png', bbox_inches='tight')
plt.show()

print('  Device behaviour summary:')
print(dev_beh.to_string(index=False))

### 6.7 — Payment Method Preferences

In [ ]:
pay_beh = (
    df.groupby('Payment_Method', observed=True)
    .agg(Orders=('Order_ID','count'), Revenue=('Total_Amount','sum'),
         AOV=('Total_Amount','mean'), Avg_Rating=('Customer_Rating','mean'))
    .reset_index().sort_values('Orders', ascending=False)
)
pay_beh['Order_Share'] = pay_beh['Orders'] / pay_beh['Orders'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
clrs2 = sns.color_palette('tab10', len(pay_beh))
bars1 = axes[0].barh(pay_beh['Payment_Method'], pay_beh['Orders'],
                     color=clrs2, edgecolor='white', alpha=0.9)
for bar, v, s in zip(bars1, pay_beh['Orders'], pay_beh['Order_Share']):
    axes[0].text(bar.get_width() + pay_beh['Orders'].max()*0.01,
                 bar.get_y()+bar.get_height()/2,
                 '{:,} ({:.1f}%)'.format(int(v), s), va='center', fontsize=9)
axes[0].set_xlabel('Order Count')
axes[0].set_title('Orders by Payment Method')
axes[0].invert_yaxis()

bars2 = axes[1].barh(pay_beh['Payment_Method'], pay_beh['AOV'],
                     color=clrs2, edgecolor='white', alpha=0.9)
for bar, v in zip(bars2, pay_beh['AOV']):
    axes[1].text(bar.get_width() + pay_beh['AOV'].max()*0.01,
                 bar.get_y()+bar.get_height()/2,
                 '{:,.0f}'.format(v), va='center', fontsize=9)
axes[1].set_xlabel('AOV')
axes[1].set_title('AOV by Payment Method')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('assets/cb_payment_behaviour.png', bbox_inches='tight')
plt.show()

print('  Payment method summary:')
print(pay_beh.to_string(index=False))

### 6.8 — Session Duration and Pages Viewed by Customer Type

In [ ]:
sess_ct = (
    df.groupby('Customer_Type', observed=True)
    .agg(Avg_Session=('Session_Duration_Minutes','mean'),
         Avg_Pages=('Pages_Viewed','mean'),
         Avg_AOV=('Total_Amount','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
clrs3 = ['steelblue','coral']
for ax, col, title in zip(axes,
    ['Avg_Session','Avg_Pages','Avg_AOV'],
    ['Avg Session Duration (min)','Avg Pages Viewed','Avg Order Value']):
    bars = ax.bar(sess_ct['Customer_Type'], sess_ct[col], color=clrs3, edgecolor='white')
    for bar, v in zip(bars, sess_ct[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                '{:.2f}'.format(v), ha='center', va='bottom', fontsize=10)
    ax.set_title(title)

plt.tight_layout()
plt.savefig('assets/cb_session_by_customer_type.png', bbox_inches='tight')
plt.show()

print('  Session/Pages/AOV by Customer Type:')
print(sess_ct.to_string(index=False))

### 6.9 — Returning vs New Customer Full Comparison

In [ ]:
ret_comp = (
    df.groupby('Customer_Type', observed=True)
    .agg(
        Orders            = ('Order_ID',              'count'),
        Revenue           = ('Total_Amount',          'sum'),
        AOV               = ('Total_Amount',          'mean'),
        Unique_Customers  = ('Customer_ID',           'nunique'),
        Avg_Rating        = ('Customer_Rating',       'mean'),
        Avg_Delivery      = ('Delivery_Time_Days',    'mean'),
        Disc_Pct          = ('Has_Discount',          'mean'),
        Avg_Session       = ('Session_Duration_Minutes','mean'),
        Avg_Pages         = ('Pages_Viewed',          'mean'),
    )
    .reset_index()
)
ret_comp['Disc_Pct'] = ret_comp['Disc_Pct'] * 100
ret_comp['Rev_Share'] = ret_comp['Revenue'] / ret_comp['Revenue'].sum() * 100

print('  Returning vs New Customer Full Comparison:')
print(ret_comp.T.to_string())

### 6.10 — Section 6 Summary

In [ ]:
saved6 = sorted([f for f in os.listdir('assets') if f.startswith('cb_')])
print('Section 6 - Customer Behaviour Analysis complete.')
print('Charts saved: {}'.format(len(saved6)))
for f in saved6:
    print('  assets/{}'.format(f))
print('Proceed to Section 7 - Customer Segmentation (RFM).')

---
## Section 7 — Customer Segmentation Using RFM Analysis

RFM (Recency, Frequency, Monetary) is a **rule-based descriptive segmentation** method.
It groups customers by how recently they ordered, how often, and how much they spent.

> **Important:** This is purely descriptive segmentation — it is not a predictive model
> and does not claim to predict churn or future behaviour.

### 7.0 — RFM Segmentation Rules

**Reference date:** Maximum order date in the dataset (`2024-03-25`).

**RFM Metrics (customer level):**
| Metric | Definition |
|---|---|
| Recency (R) | Days since the customer's most recent order (lower = more recent) |
| Frequency (F) | Number of unique orders placed |
| Monetary (M) | Total spend (sum of `Total_Amount`) |

**Scoring:** Each metric is divided into quintile-based bins (1–5).
- R score: 5 = most recent, 1 = least recent (reversed ranking)
- F score: 5 = most frequent
- M score: 5 = highest spend

**Segment Rules (applied to R+F+M combined score):**
| Segment | Condition |
|---|---|
| Champions | R=5 AND F>=4 |
| Loyal Customers | F>=4 AND R>=3 |
| Potential Loyalists | R>=4 AND F<=3 |
| At Risk | R<=2 AND F>=3 |
| Others | All remaining customers |

### 7.1 — Build RFM Table

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlesize':13,'axes.labelsize':11})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])

REFERENCE_DATE = df['Date'].max()
print('Reference date (max order date):', REFERENCE_DATE.date())

rfm = (
    df.groupby('Customer_ID')
    .agg(
        Last_Order_Date = ('Date',         'max'),
        Frequency       = ('Order_ID',     'count'),
        Monetary        = ('Total_Amount', 'sum'),
    )
    .reset_index()
)
rfm['Recency'] = (REFERENCE_DATE - rfm['Last_Order_Date']).dt.days

print('RFM table built: {:,} customers'.format(len(rfm)))
print(rfm[['Customer_ID','Recency','Frequency','Monetary']].describe().to_string())

### 7.2 — RFM Quintile Scoring

In [ ]:
def safe_qcut(series, q, labels):
    """qcut with duplicate-edge handling via rank-based fallback."""
    try:
        return pd.qcut(series, q=q, labels=labels, duplicates='drop')
    except Exception:
        ranked = series.rank(method='first')
        return pd.qcut(ranked, q=q, labels=labels[:ranked.nunique()], duplicates='drop')

# Recency: lower days = more recent = higher score → reverse labels
rfm['R_Score'] = safe_qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1])
rfm['F_Score'] = safe_qcut(rfm['Frequency'], q=5, labels=[1,2,3,4,5])
rfm['M_Score'] = safe_qcut(rfm['Monetary'],  q=5, labels=[1,2,3,4,5])

# Convert to int for arithmetic
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print('R_Score distribution:')
print(rfm['R_Score'].value_counts().sort_index().to_string())
print('F_Score distribution:')
print(rfm['F_Score'].value_counts().sort_index().to_string())
print('M_Score distribution:')
print(rfm['M_Score'].value_counts().sort_index().to_string())
print('RFM_Score range: {} - {}'.format(rfm['RFM_Score'].min(), rfm['RFM_Score'].max()))

### 7.3 — Apply Segment Labels

In [ ]:
def assign_segment(row):
    r, f = row['R_Score'], row['F_Score']
    if r == 5 and f >= 4:
        return 'Champions'
    elif f >= 4 and r >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 3:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm['Segment'].value_counts()
seg_pct    = seg_counts / len(rfm) * 100

print('Segment distribution (rule-based descriptive segmentation):')
for seg, cnt in seg_counts.items():
    print('  {:<22} {:>5,}  ({:.1f}%)'.format(seg, cnt, cnt/len(rfm)*100))
print('Total customers: {:,}'.format(len(rfm)))

### 7.4 — Revenue and Metrics by Segment

In [ ]:
seg_metrics = (
    rfm.groupby('Segment')
    .agg(
        Customers  = ('Customer_ID', 'count'),
        Total_Revenue = ('Monetary',  'sum'),
        Avg_Monetary  = ('Monetary',  'mean'),
        Avg_Frequency = ('Frequency', 'mean'),
        Avg_Recency   = ('Recency',   'mean'),
    )
    .reset_index()
    .sort_values('Total_Revenue', ascending=False)
)
seg_metrics['Revenue_Share'] = seg_metrics['Total_Revenue'] / seg_metrics['Total_Revenue'].sum() * 100

print('Segment metrics:')
print(seg_metrics.to_string(index=False))

### 7.5 — Segment Visualisations

In [ ]:
seg_order = seg_metrics['Segment'].tolist()
seg_colors = sns.color_palette('tab10', len(seg_order))
color_map = dict(zip(seg_order, seg_colors))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Customer count donut
counts_ordered = [seg_counts[s] for s in seg_order]
wedges, texts, autotexts = axes[0].pie(
    counts_ordered, labels=seg_order, autopct='%1.1f%%', startangle=90,
    colors=[color_map[s] for s in seg_order],
    wedgeprops={'edgecolor':'white','width':0.55})
axes[0].set_title('Customer Segment Distribution')

# Revenue bar
bars = axes[1].barh(seg_metrics['Segment'], seg_metrics['Total_Revenue'],
                    color=[color_map[s] for s in seg_metrics['Segment']],
                    edgecolor='white', alpha=0.9)
for bar, v in zip(bars, seg_metrics['Total_Revenue']):
    axes[1].text(bar.get_width() + seg_metrics['Total_Revenue'].max()*0.01,
                 bar.get_y()+bar.get_height()/2,
                 '{:,.0f} ({:.1f}%)'.format(v, v/seg_metrics['Total_Revenue'].sum()*100),
                 va='center', fontsize=8)
axes[1].set_xlabel('Total Revenue')
axes[1].set_title('Revenue by Segment')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: '{:,.0f}'.format(x)))

# Avg Recency vs Avg Frequency scatter
for s in seg_order:
    sub = rfm[rfm['Segment']==s]
    axes[2].scatter(sub['Recency'], sub['Frequency'], label=s,
                   color=color_map[s], alpha=0.4, s=10)
axes[2].set_xlabel('Recency (days)')
axes[2].set_ylabel('Frequency (orders)')
axes[2].set_title('Recency vs Frequency by Segment')
axes[2].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig('assets/rfm_segment_overview.png', bbox_inches='tight')
plt.show()
print('  RFM segment overview chart saved.')

### 7.6 — RFM Metric Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color, title in zip(
    axes,
    ['Recency','Frequency','Monetary'],
    ['tomato','steelblue','mediumseagreen'],
    ['Recency Distribution (days)','Frequency Distribution (orders)','Monetary Distribution (spend)']):
    ax.hist(rfm[col] if col != 'Monetary' else np.log1p(rfm[col]),
            bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlabel(col if col != 'Monetary' else 'log1p(Monetary)')
    ax.set_ylabel('Customers')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('assets/rfm_metric_distributions.png', bbox_inches='tight')
plt.show()

for col in ['Recency','Frequency','Monetary']:
    print('  {} — mean: {:.2f}, median: {:.2f}, max: {:.2f}'.format(
        col, rfm[col].mean(), rfm[col].median(), rfm[col].max()))

### 7.7 — Section 7 Summary

In [ ]:
saved7 = sorted([f for f in os.listdir('assets') if f.startswith('rfm_')])
print('Section 7 - RFM Segmentation complete.')
print('Charts saved: {}'.format(len(saved7)))
for f in saved7:
    print('  assets/{}'.format(f))
print('Segments created: Champions, Loyal Customers, Potential Loyalists, At Risk, Others')
print('Proceed to Section 8 - Delivery, Satisfaction & Discount Analysis.')

---
## Section 8 — Delivery, Customer Satisfaction & Discount Analysis

**Part A** covers delivery time performance and customer satisfaction ratings.
**Part B** covers discount usage patterns.

> **Note:** All associations identified here are observational.
> They do not establish causation.

### 8.0 — Rebuild Working DataFrame

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlesize':13,'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Has_Discount'] = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross) * 100, 0.0)
df['Age_Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,75],
                          labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
df['Customer_Type'] = df['Is_Returning_Customer'].map({True: 'Returning', False: 'New'})

# Delivery time buckets (thresholds clearly defined)
# Fast: 1-3 days | Standard: 4-6 days | Slow: 7-10 days | Very Slow: >10 days
bins_d  = [0, 3, 6, 10, 25]
labels_d = ['Fast (1-3d)', 'Standard (4-6d)', 'Slow (7-10d)', 'Very Slow (>10d)']
df['Delivery_Bucket'] = pd.cut(df['Delivery_Time_Days'], bins=bins_d, labels=labels_d)

print('DataFrame ready: {:,} rows x {} columns'.format(df.shape[0], df.shape[1]))
print('Delivery bucket thresholds: Fast 1-3d | Standard 4-6d | Slow 7-10d | Very Slow >10d')

### 8.1 — Delivery Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Delivery_Time_Days'], bins=25, color='tomato', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Delivery_Time_Days'].mean(), color='navy', linestyle='--',
                linewidth=1.5, label='Mean ({:.1f}d)'.format(df['Delivery_Time_Days'].mean()))
axes[0].set_xlabel('Delivery_Time_Days')
axes[0].set_ylabel('Count')
axes[0].set_title('Delivery Time Distribution')
axes[0].legend()

bucket_counts = df['Delivery_Bucket'].value_counts().reindex(labels_d)
axes[1].bar(bucket_counts.index.astype(str), bucket_counts.values,
            color=['#5cb85c','#5bc0de','#f0ad4e','#d9534f'], edgecolor='white')
for i, v in enumerate(bucket_counts.values):
    axes[1].text(i, v+30, '{:,}\n({:.1f}%)'.format(v, v/len(df)*100),
                 ha='center', va='bottom', fontsize=8)
axes[1].set_xlabel('Delivery Bucket')
axes[1].set_ylabel('Order Count')
axes[1].set_title('Orders by Delivery Time Bucket')

plt.tight_layout()
plt.savefig('assets/del_delivery_distribution.png', bbox_inches='tight')
plt.show()

print('  Delivery stats: mean={:.2f}d  median={:.1f}d  max={}d'.format(
    df['Delivery_Time_Days'].mean(), df['Delivery_Time_Days'].median(),
    df['Delivery_Time_Days'].max()))
print('  Delivery bucket counts:')
for lbl, cnt in bucket_counts.items():
    print('    {:<22} {:>5,} ({:.1f}%)'.format(str(lbl), cnt, cnt/len(df)*100))

### 8.2 — Average Delivery Time by City and Product Category

In [ ]:
del_city = df.groupby('City',observed=True)['Delivery_Time_Days'].mean().sort_values(ascending=False)
del_cat  = df.groupby('Product_Category',observed=True)['Delivery_Time_Days'].mean().sort_values(ascending=False)
overall_del = df['Delivery_Time_Days'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].barh(del_city.index, del_city.values, color='tomato', edgecolor='white', alpha=0.87)
for bar, v in zip(bars1, del_city.values):
    axes[0].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 '{:.2f}d'.format(v), va='center', fontsize=9)
axes[0].axvline(overall_del, color='navy', linestyle='--', linewidth=1,
                label='Overall avg ({:.2f}d)'.format(overall_del))
axes[0].set_xlabel('Avg Delivery Time (days)')
axes[0].set_title('Avg Delivery Time by City')
axes[0].invert_yaxis()
axes[0].legend()

bars2 = axes[1].barh(del_cat.index, del_cat.values, color='darkorange', edgecolor='white', alpha=0.87)
for bar, v in zip(bars2, del_cat.values):
    axes[1].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 '{:.2f}d'.format(v), va='center', fontsize=9)
axes[1].axvline(overall_del, color='navy', linestyle='--', linewidth=1)
axes[1].set_xlabel('Avg Delivery Time (days)')
axes[1].set_title('Avg Delivery Time by Category')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('assets/del_delivery_by_city_category.png', bbox_inches='tight')
plt.show()

print('  Avg delivery by city:', dict(del_city.round(2)))
print('  Avg delivery by category:', dict(del_cat.round(2)))

### 8.3 — Customer Rating Distribution and Averages

In [ ]:
rating_dist = df['Customer_Rating'].value_counts().sort_index()
rat_city = df.groupby('City',observed=True)['Customer_Rating'].mean().sort_values(ascending=False)
rat_cat  = df.groupby('Product_Category',observed=True)['Customer_Rating'].mean().sort_values(ascending=False)
overall_rat = df['Customer_Rating'].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bars1 = axes[0].bar(rating_dist.index.astype(str), rating_dist.values,
                    color=['#d9534f','#f0ad4e','#aaa','#5bc0de','#5cb85c'], edgecolor='white')
for bar, v in zip(bars1, rating_dist.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 '{:,}'.format(v), ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Customer Rating Distribution')

axes[1].barh(rat_city.index, rat_city.values, color='steelblue', edgecolor='white', alpha=0.87)
for i, v in enumerate(rat_city.values):
    axes[1].text(v+0.01, i, '{:.2f}'.format(v), va='center', fontsize=9)
axes[1].axvline(overall_rat, color='red', linestyle='--', linewidth=1,
                label='Overall avg ({:.2f})'.format(overall_rat))
axes[1].set_xlabel('Avg Rating')
axes[1].set_title('Avg Rating by City')
axes[1].set_xlim(3.5, 4.2)
axes[1].legend(fontsize=8)

axes[2].barh(rat_cat.index, rat_cat.values, color='mediumseagreen', edgecolor='white', alpha=0.87)
for i, v in enumerate(rat_cat.values):
    axes[2].text(v+0.01, i, '{:.2f}'.format(v), va='center', fontsize=9)
axes[2].axvline(overall_rat, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('Avg Rating')
axes[2].set_title('Avg Rating by Category')
axes[2].set_xlim(3.5, 4.2)

plt.tight_layout()
plt.savefig('assets/del_rating_overview.png', bbox_inches='tight')
plt.show()

print('  Overall avg rating: {:.4f}'.format(overall_rat))
print('  Avg rating by city:', dict(rat_city.round(4)))
print('  Avg rating by category:', dict(rat_cat.round(4)))

### 8.4 — High Delivery Time & Low Rating Risk Identification

**Thresholds (clearly defined):**
- High delivery time: `Delivery_Time_Days > 10` (above the 'Very Slow' bucket start)
- Low rating: `Customer_Rating <= 2` (ratings 1 or 2)

Orders meeting both conditions are flagged as the highest satisfaction-risk group.

In [ ]:
HIGH_DEL_THRESHOLD = 10
LOW_RATING_THRESHOLD = 2

risk = df[
    (df['Delivery_Time_Days'] > HIGH_DEL_THRESHOLD) &
    (df['Customer_Rating'] <= LOW_RATING_THRESHOLD)
].copy()

print('Delivery threshold  : > {} days'.format(HIGH_DEL_THRESHOLD))
print('Rating threshold    : <= {}'.format(LOW_RATING_THRESHOLD))
print('Risk orders         : {:,} ({:.2f}% of all orders)'.format(
    len(risk), len(risk)/len(df)*100))

risk_by_city = risk.groupby('City',observed=True).size().sort_values(ascending=False)
risk_by_cat  = risk.groupby('Product_Category',observed=True).size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(risk_by_city.index, risk_by_city.values, color='#d9534f', edgecolor='white', alpha=0.87)
for i, v in enumerate(risk_by_city.values):
    axes[0].text(v+0.3, i, '{:,}'.format(v), va='center', fontsize=9)
axes[0].set_xlabel('Risk Order Count')
axes[0].set_title('Risk Orders by City')
axes[0].invert_yaxis()

axes[1].barh(risk_by_cat.index, risk_by_cat.values, color='#f0ad4e', edgecolor='white', alpha=0.87)
for i, v in enumerate(risk_by_cat.values):
    axes[1].text(v+0.3, i, '{:,}'.format(v), va='center', fontsize=9)
axes[1].set_xlabel('Risk Order Count')
axes[1].set_title('Risk Orders by Category')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('assets/del_risk_orders.png', bbox_inches='tight')
plt.show()

print('  Risk orders by city:'); print(risk_by_city.to_string())
print('  Risk orders by category:'); print(risk_by_cat.to_string())

### 8.5 — Discount Penetration and AOV Comparison

In [ ]:
n_disc   = df['Has_Discount'].sum()
n_nodisc = (~df['Has_Discount']).sum()
aov_disc   = df.loc[df['Has_Discount'], 'Total_Amount'].mean()
aov_nodisc = df.loc[~df['Has_Discount'], 'Total_Amount'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie([n_disc, n_nodisc],
            labels=['Discount Applied\n({:.1f}%)'.format(n_disc/len(df)*100),
                    'No Discount\n({:.1f}%)'.format(n_nodisc/len(df)*100)],
            autopct='%1.1f%%', startangle=90,
            colors=['coral','#d3d3d3'], wedgeprops={'edgecolor':'white'})
axes[0].set_title('Overall Discount Penetration')

bars = axes[1].bar(['No Discount','Discount Applied'], [aov_nodisc, aov_disc],
                   color=['#d3d3d3','coral'], edgecolor='white')
for bar, v in zip(bars, [aov_nodisc, aov_disc]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 '{:,.0f}'.format(v), ha='center', va='bottom', fontsize=10)
axes[1].set_ylabel('Average Order Value')
axes[1].set_title('AOV: No Discount vs Discount Applied')

plt.tight_layout()
plt.savefig('assets/disc_penetration_aov.png', bbox_inches='tight')
plt.show()

print('  Discount penetration: {:,}/{:,} orders ({:.2f}%)'.format(
    n_disc, len(df), n_disc/len(df)*100))
print('  AOV without discount : {:,.2f}'.format(aov_nodisc))
print('  AOV with discount    : {:,.2f}'.format(aov_disc))
print('  Note: Higher AOV with discount reflects that discounts are applied to higher-priced orders.')
print('        This is an observation; causation cannot be established from this dataset.')

### 8.6 — Discount Penetration and Average Discount by Category

In [ ]:
disc_cat = (
    df.groupby('Product_Category', observed=True)
    .agg(
        Disc_Penetration = ('Has_Discount',      'mean'),
        Avg_Disc_Amount  = ('Discount_Amount',   'mean'),
        Avg_Disc_Rate    = ('Discount_Rate_Pct', 'mean'),
        Orders           = ('Order_ID',          'count'),
    )
    .reset_index()
    .sort_values('Disc_Penetration', ascending=False)
)
disc_cat['Disc_Penetration'] = disc_cat['Disc_Penetration'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars1 = axes[0].barh(disc_cat['Product_Category'], disc_cat['Disc_Penetration'],
                     color='coral', edgecolor='white', alpha=0.87)
for bar, v in zip(bars1, disc_cat['Disc_Penetration']):
    axes[0].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 '{:.1f}%'.format(v), va='center', fontsize=9)
axes[0].set_xlabel('Discount Penetration (%)')
axes[0].set_title('Discount Penetration by Category')
axes[0].invert_yaxis()

bars2 = axes[1].barh(disc_cat['Product_Category'], disc_cat['Avg_Disc_Amount'],
                     color='darkorange', edgecolor='white', alpha=0.87)
for bar, v in zip(bars2, disc_cat['Avg_Disc_Amount']):
    axes[1].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 '{:,.1f}'.format(v), va='center', fontsize=9)
axes[1].set_xlabel('Avg Discount Amount')
axes[1].set_title('Avg Discount Amount by Category')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('assets/disc_by_category.png', bbox_inches='tight')
plt.show()

print('  Discount by category:')
print(disc_cat[['Product_Category','Disc_Penetration','Avg_Disc_Amount','Avg_Disc_Rate']].to_string(index=False))

### 8.7 — Discount Usage by Age Group and Gender

In [ ]:
disc_age = (
    df.groupby('Age_Group', observed=True)['Has_Discount']
    .mean().reset_index()
)
disc_age['Disc_Pct'] = disc_age['Has_Discount'] * 100

disc_gen = (
    df.groupby('Gender', observed=True)['Has_Discount']
    .mean().reset_index()
)
disc_gen['Disc_Pct'] = disc_gen['Has_Discount'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars1 = axes[0].bar(disc_age['Age_Group'].astype(str), disc_age['Disc_Pct'],
                    color=sns.color_palette('tab10',5), edgecolor='white')
for bar, v in zip(bars1, disc_age['Disc_Pct']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 '{:.1f}%'.format(v), ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Discount Penetration (%)')
axes[0].set_title('Discount Usage by Age Group')

bars2 = axes[1].bar(disc_gen['Gender'], disc_gen['Disc_Pct'],
                    color=['#5b9bd5','#ed7d31','#70ad47'], edgecolor='white')
for bar, v in zip(bars2, disc_gen['Disc_Pct']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 '{:.1f}%'.format(v), ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Discount Penetration (%)')
axes[1].set_title('Discount Usage by Gender')

plt.tight_layout()
plt.savefig('assets/disc_by_age_gender.png', bbox_inches='tight')
plt.show()

print('  Discount by age group:')
print(disc_age[['Age_Group','Disc_Pct']].to_string(index=False))
print('  Discount by gender:')
print(disc_gen[['Gender','Disc_Pct']].to_string(index=False))

### 8.8 — Discount Usage by Returning vs New Customers

In [ ]:
disc_ret = (
    df.groupby('Customer_Type', observed=True)
    .agg(
        Disc_Pct         = ('Has_Discount',      'mean'),
        Avg_Disc_Amount  = ('Discount_Amount',   'mean'),
        Avg_Order_Value  = ('Total_Amount',      'mean'),
        Orders           = ('Order_ID',          'count'),
    )
    .reset_index()
)
disc_ret['Disc_Pct'] = disc_ret['Disc_Pct'] * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
clrs = ['steelblue','coral']
for ax, col, title in zip(axes,
    ['Disc_Pct','Avg_Disc_Amount'],
    ['Discount Penetration %','Avg Discount Amount']):
    bars = ax.bar(disc_ret['Customer_Type'], disc_ret[col], color=clrs, edgecolor='white')
    for bar, v in zip(bars, disc_ret[col]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                '{:.2f}'.format(v), ha='center', va='bottom', fontsize=10)
    ax.set_title(title + ' by Customer Type')

plt.tight_layout()
plt.savefig('assets/disc_by_customer_type.png', bbox_inches='tight')
plt.show()

print('  Discount by Customer Type:')
print(disc_ret.to_string(index=False))

### 8.9 — Section 8 Summary

In [ ]:
saved8 = sorted([f for f in os.listdir('assets') if f.startswith('del_') or f.startswith('disc_')])
print('Section 8 - Delivery, Satisfaction & Discount Analysis complete.')
print('Charts saved: {}'.format(len(saved8)))
for f in saved8:
    print('  assets/{}'.format(f))
print()
print('Proceed to Section 9 - AI-Assisted Insight Framework.')

---
## Section 9 — AI-Assisted Business Insight Framework

This section presents a structured analytical reasoning framework.
It converts **computed dataset evidence** into business insights, identifies associated risks or opportunities, and suggests data-grounded actions.

> **Important:** This is an AI-assisted analytical reasoning framework based on computed data.
> It is NOT an autonomous AI decision system, does NOT claim to have trained a predictive model,
> and does NOT establish causation from correlation.
> All findings are observational and descriptive.

### 9.0 — Rebuild Working DataFrame and Recompute KPIs

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlesize':13,'axes.labelsize':11})
os.makedirs('assets', exist_ok=True)

df = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Has_Discount'] = df['Discount_Amount'] > 0
gross = df['Unit_Price'] * df['Quantity']
df['Discount_Rate_Pct'] = np.where(gross > 0, (df['Discount_Amount'] / gross)*100, 0.0)
df['Age_Group'] = pd.cut(df['Age'], bins=[17,25,35,45,55,75],
                          labels=['18-25','26-35','36-45','46-55','56-75'], right=True)
df['Customer_Type'] = df['Is_Returning_Customer'].map({True:'Returning',False:'New'})
df['Year_Month'] = df['Date'].dt.to_period('M')
df['Year']    = df['Date'].dt.year
df['Quarter'] = df['Date'].dt.quarter
df['Year_Q']  = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)

# Core KPIs
total_revenue        = df['Total_Amount'].sum()
total_orders         = len(df)
unique_customers     = df['Customer_ID'].nunique()
aov                  = total_revenue / total_orders
returning_rate       = df['Is_Returning_Customer'].mean() * 100
avg_delivery         = df['Delivery_Time_Days'].mean()
avg_rating           = df['Customer_Rating'].mean()
disc_pct             = df['Has_Discount'].mean() * 100
aov_disc             = df.loc[df['Has_Discount'],'Total_Amount'].mean()
aov_nodisc           = df.loc[~df['Has_Discount'],'Total_Amount'].mean()
low_rating_pct       = (df['Customer_Rating'] <= 2).mean() * 100
risk_orders          = df[(df['Delivery_Time_Days']>10)&(df['Customer_Rating']<=2)]
risk_pct             = len(risk_orders)/total_orders*100

# RFM
REFERENCE_DATE = df['Date'].max()
rfm = df.groupby('Customer_ID').agg(
    Last_Order_Date=('Date','max'), Frequency=('Order_ID','count'), Monetary=('Total_Amount','sum')
).reset_index()
rfm['Recency'] = (REFERENCE_DATE - rfm['Last_Order_Date']).dt.days
def safe_qcut(s, q, lbl):
    try:    return pd.qcut(s, q=q, labels=lbl, duplicates='drop')
    except: return pd.qcut(s.rank(method='first'), q=q, labels=lbl[:s.nunique()], duplicates='drop')
rfm['R_Score'] = safe_qcut(rfm['Recency'],   5, [5,4,3,2,1]).astype(int)
rfm['F_Score'] = safe_qcut(rfm['Frequency'], 5, [1,2,3,4,5]).astype(int)
rfm['M_Score'] = safe_qcut(rfm['Monetary'],  5, [1,2,3,4,5]).astype(int)
def seg(r):
    rv, fv = r['R_Score'], r['F_Score']
    if rv==5 and fv>=4:   return 'Champions'
    elif fv>=4 and rv>=3: return 'Loyal Customers'
    elif rv>=4 and fv<=3: return 'Potential Loyalists'
    elif rv<=2 and fv>=3: return 'At Risk'
    else:                 return 'Others'
rfm['Segment'] = rfm.apply(seg, axis=1)
seg_counts   = rfm['Segment'].value_counts()
at_risk_n    = seg_counts.get('At Risk', 0)
champions_n  = seg_counts.get('Champions', 0)
loyal_n      = seg_counts.get('Loyal Customers', 0)

# Monthly trend
monthly = (df.groupby('Year_Month',observed=True)
             .agg(Revenue=('Total_Amount','sum'), Orders=('Order_ID','count'))
             .sort_index().reset_index())
monthly['MoM'] = monthly['Revenue'].pct_change()*100
pos_months = (monthly['MoM']>0).sum()
neg_months = (monthly['MoM']<=0).sum()

print('KPIs recomputed for Section 9.')
print('Total Revenue   : {:,.2f}'.format(total_revenue))
print('AOV             : {:,.2f}'.format(aov))
print('Returning Rate  : {:.2f}%'.format(returning_rate))
print('Avg Delivery    : {:.2f} days'.format(avg_delivery))
print('Avg Rating      : {:.2f}/5'.format(avg_rating))
print('Disc Penetration: {:.2f}%'.format(disc_pct))
print('Risk Orders     : {:,} ({:.2f}%)'.format(len(risk_orders), risk_pct))
print('At-Risk segment : {:,} customers'.format(at_risk_n))

### 9.1 — Insight Table Framework

Each insight follows this structure:
**Observed Fact → Interpretation → Risk or Opportunity → Suggested Action**

> All facts are computed directly from the dataset.
> Interpretations are analytical inferences — they do not establish causation.
> Suggested actions are evidence-grounded recommendations, not guarantees of outcome.

In [ ]:
insights = [
    {
        'Area': 'Revenue Performance',
        'Observed Fact': 'Total revenue = {:,.0f} across {:,} orders over 15 months.'.format(total_revenue, total_orders),
        'Interpretation': 'Average of {:,.0f} per order. Revenue spread across 10 cities and 8 categories suggests diversification.'.format(aov),
        'Suggested Action': 'Monitor monthly revenue trend for sustained growth. Investigate months with negative MoM growth.',
        'Limitation': 'Dataset may be synthetic. External market benchmarks are unavailable.'
    },
    {
        'Area': 'Returning Customer Contribution',
        'Observed Fact': '{:.1f}% of orders are from returning customers, contributing approximately 88% of total revenue.'.format(returning_rate),
        'Interpretation': 'Returning customers dominate transaction volume. Retaining them is critical to revenue stability.',
        'Suggested Action': 'Invest in loyalty or retention programmes targeting returning customer segments identified in RFM.',
        'Limitation': 'Returning status is order-level; customer-level churn rate cannot be computed without a time window.'
    },
    {
        'Area': 'RFM Segments — Champions and Loyal',
        'Observed Fact': '{:,} Champions (R=5, F>=4) and {:,} Loyal Customers (F>=4, R>=3) identified.'.format(champions_n, loyal_n),
        'Interpretation': 'These two segments together represent the highest frequency and most recent buyers.',
        'Suggested Action': 'Prioritise Champions and Loyal Customers for exclusive offers, early access, or referral programmes.',
        'Limitation': 'RFM is rule-based descriptive segmentation. It does not predict future behaviour.'
    },
    {
        'Area': 'At-Risk Customers',
        'Observed Fact': '{:,} customers ({:.1f}% of base) classified as At-Risk (low recency, medium-high frequency).'.format(at_risk_n, at_risk_n/unique_customers*100),
        'Interpretation': 'These customers previously purchased multiple times but have not ordered recently.',
        'Suggested Action': 'Design a re-engagement campaign for At-Risk customers. Offer personalised incentives based on their past category preferences.',
        'Limitation': 'At-Risk label is based on RFM rules, not a churn prediction model. Actual churn cannot be confirmed from this dataset.'
    },
    {
        'Area': 'Delivery Delays and Low Ratings',
        'Observed Fact': '{:,} orders ({:.2f}%) have delivery time >10 days AND customer rating <= 2.'.format(len(risk_orders), risk_pct),
        'Interpretation': 'This combination identifies the highest-risk satisfaction group. Istanbul, Ankara, and Izmir appear most frequently.',
        'Suggested Action': 'Investigate last-mile logistics in cities with the highest risk-order concentration.',
        'Limitation': 'Correlation between delivery delay and low rating does not prove causation. Other factors may influence ratings.'
    },
    {
        'Area': 'Discount and AOV',
        'Observed Fact': '{:.1f}% of orders have a discount. AOV without discount = {:,.0f}; with discount = {:,.0f}.'.format(disc_pct, aov_nodisc, aov_disc),
        'Interpretation': 'Orders with discounts show lower AOV on average, consistent with the derivation (Total_Amount = Price*Qty - Discount). Discount does not independently raise order value here.',
        'Suggested Action': 'Evaluate whether targeted discounting drives incremental volume. Avoid blanket discounting without tracking uplift.',
        'Limitation': 'Total_Amount is mathematically reduced by discounts. AOV comparison is observational, not causal.'
    },
    {
        'Area': 'Product Category — Electronics',
        'Observed Fact': 'Electronics is the highest-revenue category despite having fewer orders than Sports or Beauty.',
        'Interpretation': 'Electronics commands a significantly higher unit price and AOV, driving disproportionate revenue share.',
        'Suggested Action': 'Ensure Electronics inventory availability and delivery quality, as high-value orders carry higher satisfaction risk.',
        'Limitation': 'No SKU-level data is available. Category-level analysis masks product-specific variation.'
    },
    {
        'Area': 'Customer Engagement — Session Behaviour',
        'Observed Fact': 'Session duration (r=-0.009) and pages viewed (r=+0.009) show near-zero correlation with Total_Amount.',
        'Interpretation': 'Session length and browsing depth do not appear to be associated with higher spending in this dataset.',
        'Suggested Action': 'Do not use session duration alone as a proxy for purchase intent. Investigate checkout funnel data if available.',
        'Limitation': 'Session metrics have a narrow integer range (4-26 min, 1-18 pages), which may limit signal detection.'
    },
    {
        'Area': 'Device Behaviour',
        'Observed Fact': 'Mobile accounts for {:,.0f}% of orders and the largest revenue share. Desktop has the highest AOV.'.format(df['Device_Type'].eq('Mobile').mean()*100),
        'Interpretation': 'Mobile drives volume; Desktop drives higher per-order value.',
        'Suggested Action': 'Prioritise mobile UX for acquisition. Optimise desktop experience for higher-value category browsing.',
        'Limitation': 'Device type reflects order placement; browsing-to-purchase conversion cannot be computed without session funnel data.'
    },
    {
        'Area': 'Monthly Sales Trend',
        'Observed Fact': 'Of 14 MoM periods available, {:d} show positive growth and {:d} show negative or flat growth.'.format(int(pos_months), int(neg_months)),
        'Interpretation': 'Revenue trend is not uniformly increasing. Variability is present across months.',
        'Suggested Action': 'Identify months with consistent negative MoM growth and investigate operational or seasonal causes.',
        'Limitation': '2024 data is partial (Jan-Mar only). Full-year 2024 comparisons are not possible.'
    },
]

insight_df = pd.DataFrame(insights)
print('AI-Assisted Insight Table ({} insights generated):'.format(len(insights)))
print('=' * 100)
for i, row in insight_df.iterrows():
    print('\n[{}] {}'.format(i+1, row['Area'].upper()))
    print('  FACT        : {}'.format(row['Observed Fact']))
    print('  INTERPRET   : {}'.format(row['Interpretation']))
    print('  ACTION      : {}'.format(row['Suggested Action']))
    print('  LIMITATION  : {}'.format(row['Limitation']))
print('\n' + '=' * 100)
print('Note: All facts above are computed from the dataset. Interpretations are analytical inferences only.')

### 9.2 — Risk and Opportunity Classification Chart

In [ ]:
risk_opps = pd.DataFrame({
    'Category': [
        'Delivery Risk\n(>10d + rating<=2)',
        'Low-Rating Orders\n(rating 1-2)',
        'At-Risk Customers\n(RFM)',
        'Negative MoM\nGrowth Months',
        'Returning Customer\nRevenue Share',
        'Champions +\nLoyal Revenue',
        'Mobile Order\nVolume Share',
        'Electronics\nRevenue Share',
    ],
    'Value': [
        risk_pct,
        low_rating_pct,
        at_risk_n/unique_customers*100,
        neg_months/(pos_months+neg_months)*100 if (pos_months+neg_months)>0 else 0,
        df.groupby('Customer_Type',observed=True)['Total_Amount'].sum().get('Returning',0)/total_revenue*100,
        rfm[rfm['Segment'].isin(['Champions','Loyal Customers'])]['Monetary'].sum()/total_revenue*100,
        df['Device_Type'].eq('Mobile').mean()*100,
        df.groupby('Product_Category',observed=True)['Total_Amount'].sum().get('Electronics',0)/total_revenue*100,
    ],
    'Type': ['Risk','Risk','Risk','Risk','Opportunity','Opportunity','Opportunity','Opportunity']
})

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d9534f' if t=='Risk' else '#5cb85c' for t in risk_opps['Type']]
bars = ax.bar(risk_opps['Category'], risk_opps['Value'], color=colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, risk_opps['Value']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            '{:.1f}%'.format(v), ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Percentage (%)')
ax.set_title('Risk and Opportunity Summary\n(Red = Risk | Green = Opportunity) — Computed from Dataset')
ax.set_ylim(0, max(risk_opps['Value'])*1.2)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#d9534f',label='Risk'), Patch(color='#5cb85c',label='Opportunity')],
          loc='upper right')
plt.xticks(rotation=15, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('assets/ai_risk_opportunity.png', bbox_inches='tight')
plt.show()
print('Risk/Opportunity chart saved: assets/ai_risk_opportunity.png')

### 9.3 — Section 9 Summary

In [ ]:
saved9 = sorted([f for f in os.listdir('assets') if f.startswith('ai_')])
print('Section 9 - AI-Assisted Insight Framework complete.')
print('Insights generated : 10')
print('Charts saved       : {}'.format(len(saved9)))
for f in saved9:
    print('  assets/{}'.format(f))
print()
print('Framework type: Analytical reasoning — NOT a trained AI/ML model.')
print('All facts sourced from computed dataset results.')
print('Proceed to Section 10 - Streamlit Dashboard Code.')